Fig-1

In [ ]:
#(a)
'''
import os
from pathlib import Path
import pandas as pd
import pygmt


# =========================================================
# 1. Global configuration
# =========================================================
REGION = [-119.0999, -116.0993, 34.2695, 37.2695]
PROJECTION = "M15c"

EQ_CSV = Path("D:/a/master/Earthquake-US/Fig-Use/Fig-1/ridgecrest_usgs.csv")
STATION_FOLDER = Path("D:/a/master/Earthquake-US/Data/20190706-102/PosData-7")
DEM_GRID = Path("D:/a/master/Earthquake-US/Fig-Use/Fig-1/DEM/SRTM_Ridgecrest_30m.tif")
FAULT_SHP = Path("D:/a/master/Earthquake-US/Fig-Use/Fig-1/faults/Qfaults_US_Database.shp")

OUTPUT_PNG = Path("D:/a/master/Earthquake-US/Fig-Use-2/Fig-1/102_with_stations_gnss_major_final_v2.png")
OUTPUT_PDF = Path("D:/a/master/Earthquake-US/Fig-Use-2/Fig-1/102_with_stations_gnss_major_final_v2.pdf")

DEPTH_RANGE = [0, 10]
MAG_MIN = 1.5
SIZE_SCALE = 0.020

PLOT_BACKGROUND_EQ = True

# -----------------------------
# Colors
# -----------------------------
MAIN_64_COLOR = "#900000"
MAIN_71_COLOR = "black"
GNSS_COLOR = "#432818"

# Mainshocks
MAINSHOCKS = [
    {
        "lon": -117.504,
        "lat": 35.705,
        "depth": 8.0,
        "style": "a0.6c",
        "fill": MAIN_64_COLOR,
        "pen": None,
        "label": "Mw 6.4 Depth 8.0 km",
    },
    {
        "lon": -117.599,
        "lat": 35.769,
        "depth": 10.5,
        "style": "a0.8c",
        "fill": MAIN_71_COLOR,
        "pen": None, 
        "label": "Mw 7.1 Depth 10.5 km",
    },
]


# =========================================================
# 2. Utilities
# =========================================================
def check_file_exists(path_obj, description="file"):
    if not path_obj.exists():
        raise FileNotFoundError(f"{description} not found: {path_obj}")


def load_earthquake_catalog(csv_path, depth_range=(0, 10), mag_min=1.5, size_scale=0.015):
    check_file_exists(csv_path, "Earthquake catalog")

    df = pd.read_csv(csv_path)

    required_cols = ["longitude", "latitude", "depth", "mag"]
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in earthquake catalog: {missing}")

    df = df[required_cols].dropna()
    df = df[
        (df["depth"] >= depth_range[0]) &
        (df["depth"] <= depth_range[1]) &
        (df["mag"] >= mag_min)
    ].copy()

    df["size"] = size_scale * df["mag"]
    return df


def load_station_positions(folder_path):
    check_file_exists(folder_path, "Station folder")

    station_lons = []
    station_lats = []

    csv_files = sorted(folder_path.glob("*.csv"))
    if not csv_files:
        print(f"No station CSV files found in: {folder_path}")
        return station_lons, station_lats

    for file_path in csv_files:
        try:
            df = pd.read_csv(file_path)

            if df.empty:
                continue

            required_cols = ["NLat", "Elong"]
            if not all(col in df.columns for col in required_cols):
                print(f"Skipped {file_path.name}: missing columns {required_cols}")
                continue

            station_lats.append(df.iloc[0]["NLat"])
            station_lons.append(df.iloc[0]["Elong"])

        except Exception as e:
            print(f"Error reading {file_path.name}: {e}")

    return station_lons, station_lats


# =========================================================
# 3. Plotting
# =========================================================
def create_map_figure(eq_df, station_lons, station_lats):
    fig = pygmt.Figure()

    # -----------------------------
    # Global style
    # -----------------------------
    pygmt.config(
        FONT="15p,Helvetica,black",
        FONT_ANNOT_PRIMARY="15p,Helvetica,black",
        FONT_LABEL="15p,Helvetica,black",
        FONT_TITLE="15p,Helvetica,black",
        MAP_FRAME_TYPE="fancy",
        MAP_FRAME_PEN="0.7p,black",
        MAP_TICK_PEN_PRIMARY="0.6p,black",
        MAP_TICK_LENGTH_PRIMARY="0.14c",
        MAP_LABEL_OFFSET="0.10c",
        MAP_ANNOT_OFFSET_PRIMARY="0.07c",
        FORMAT_GEO_MAP="dddF",
    )

    # -----------------------------
    # Background topography (paper-style shaded relief)
    # -----------------------------
    pygmt.makecpt(cmap="grayC", series=[-500, 2500], reverse=False)
    
    # base relief
    fig.grdimage(
        grid=str(DEM_GRID),
        region=REGION,
        projection=PROJECTION,
        cmap=True,
        shading="+a315+nt0.8",
        transparency=8,
    )
    fig.coast(
    region=REGION,
    projection=PROJECTION,
    land="245/245/245@25",
    water="245/245/245@25",
    frame=False
    )

    # -----------------------------
    # Faults (weakened) 
    # -----------------------------
    fig.plot(
        data=str(FAULT_SHP),
        pen="0.12p,gray35@12"
    )

    # -----------------------------
    # CPT only for background earthquakes
    # -----------------------------
    pygmt.makecpt(cmap="hot", series=DEPTH_RANGE, reverse=True, background=True)

    # -----------------------------
    # Background earthquakes 
    # 稍微放大一点，大小仍表示震级
    # -----------------------------
    if PLOT_BACKGROUND_EQ and len(eq_df) > 0:
        fig.plot(
            x=eq_df["longitude"],
            y=eq_df["latitude"],
            style="c",
            size=eq_df["size"] * 0.90,
            fill=eq_df["depth"],
            cmap=True,
            pen=None,
            transparency=22,
        )

    # -----------------------------
    # GNSS stations
    # -----------------------------
    if station_lons and station_lats:
        fig.plot(
            x=station_lons,
            y=station_lats,
            style="t0.34c",
            pen=f"1.05p,{GNSS_COLOR}",
        )
        print(f"Plotted {len(station_lons)} station positions")
    else:
        print("No station positions found to plot")

    # -----------------------------
    # Mainshocks
    # 固定颜色，不参与 colorbar
    # -----------------------------
    for shock in MAINSHOCKS:
        fig.plot(
            x=[shock["lon"]],
            y=[shock["lat"]],
            style=shock["style"],
            fill=shock["fill"],
            pen=shock["pen"],
        )

    # -----------------------------
    # Basemap
    # -----------------------------
    fig.basemap(
        region=REGION,
        projection=PROJECTION,
        frame=["WSen", "xa1f0.5", "ya1f0.5"]
    )

    # -----------------------------
    # Legend
    # -----------------------------
    dummy_x = REGION[0] - 10
    dummy_y = REGION[2] - 10

    fig.plot(
        x=[dummy_x],
        y=[dummy_y],
        style="t0.34c",
        pen=f"1.05p,{GNSS_COLOR}",
        label="GNSS Stations"
    )

    fig.plot(
        x=[dummy_x],
        y=[dummy_y],
        style="a0.40c",
        fill=MAIN_64_COLOR,
        pen=None,
        label="Mw 6.4 Depth 8.0 km"
    )

    fig.plot(
        x=[dummy_x],
        y=[dummy_y],
        style="a0.40c",
        fill=MAIN_71_COLOR,
        pen=None,
        label="Mw 7.1 Depth 10.5 km"
    )

    fig.legend(
        position="JTR+jTR+o0.20c",
        box=False,
    )

    # -----------------------------
    # Colorbar
    # only for small earthquakes
    # label above the bar
    # -----------------------------
    if PLOT_BACKGROUND_EQ and len(eq_df) > 0:
        with pygmt.config(
            FONT="13p,Helvetica,black",
            FONT_LABEL="20p,Helvetica,black",
            FONT_ANNOT_PRIMARY="13p,Helvetica,black"
        ):
            fig.colorbar(
                cmap=True,
                frame=["x2+lDepth\\040(km)"],
                position="jTR+o1.25c/2.65c+w5.2c/0.42c+v",
                box=False
            )

    # -----------------------------
    # Scale bar
    # -----------------------------
    with pygmt.config(
        FONT_ANNOT_PRIMARY="11p,Helvetica,black",
        FONT_LABEL="12p,Helvetica,black"
    ):
        fig.basemap(
            map_scale="jTR+w20k+o2.25c/3.65c+f+lkm"
        )

    # -----------------------------
    # Inset map
    # -----------------------------
    # -----------------------------
    # Inset map
    # 顶刊风格：低饱和陆地绿 + 浅海蓝
    # -----------------------------
    with fig.inset(
        position="jTL+w3.8c+o0.18c",
        box="+p0.55p,black+gwhite"
    ):
        inset_region = [-124.5, -112.5, 31.8, 41.5]

        fig.coast(
            region=inset_region,
            projection="M3.8c",
            land="#cfdcc8",          # 低饱和浅绿色（陆地）
            water="#dbeaf4",         # 浅蓝色（海洋）
            borders="1/0.35p,gray45",
            shorelines="0.35p,gray45",
            frame=False
        )

        # 弱化断层，避免小地图过于杂乱
        fig.plot(
            data=str(FAULT_SHP),
            pen="0.10p,gray40@25"
        )

        # 主图范围框
        x_box = [REGION[0], REGION[1], REGION[1], REGION[0], REGION[0]]
        y_box = [REGION[2], REGION[2], REGION[3], REGION[3], REGION[2]]
        fig.plot(
            x=x_box,
            y=y_box,
            pen="0.90p,#b22222"
        )

        with pygmt.config(
            MAP_FRAME_TYPE="plain",
            MAP_FRAME_PEN="0.55p,black"
        ):
            fig.basemap(
                region=inset_region,
                projection="M3.8c",
                frame=True
            )

    return fig


# =========================================================
# 4. Main
# =========================================================
def main():
    check_file_exists(EQ_CSV, "Earthquake catalog")
    check_file_exists(STATION_FOLDER, "Station folder")
    check_file_exists(DEM_GRID, "DEM grid")
    check_file_exists(FAULT_SHP, "Fault shapefile")

    eq_df = load_earthquake_catalog(
        csv_path=EQ_CSV,
        depth_range=DEPTH_RANGE,
        mag_min=MAG_MIN,
        size_scale=SIZE_SCALE
    )
    print(f"Loaded {len(eq_df)} earthquakes after filtering")

    station_lons, station_lats = load_station_positions(STATION_FOLDER)
    print(f"Found {len(station_lons)} station positions")

    fig = create_map_figure(eq_df, station_lons, station_lats)

    OUTPUT_PNG.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(str(OUTPUT_PNG), dpi=600)
    fig.savefig(str(OUTPUT_PDF))

    print("Figure saved to:")
    print(OUTPUT_PNG)
    print(OUTPUT_PDF)


if __name__ == "__main__":
    main()
'''

In [ ]:
#(b)
'''
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import MultipleLocator
import math


def set_plot_style():
    plt.rcParams['font.family'] = 'Arial'
    plt.rcParams['font.size'] = 8
    plt.rcParams['axes.labelsize'] = 11
    plt.rcParams['xtick.labelsize'] = 10
    plt.rcParams['ytick.labelsize'] = 10
    plt.rcParams['legend.fontsize'] = 10
    plt.rcParams['pdf.fonttype'] = 42
    plt.rcParams['ps.fonttype'] = 42

    plt.rcParams['hatch.linewidth'] = 0.7


def load_and_filter_data(csv_path, start_date, end_date):
    df = pd.read_csv(csv_path)
    df['YYYYMMDD'] = pd.to_datetime(df['YYYYMMDD'], format='%Y%m%d')
    df.set_index('YYYYMMDD', inplace=True)
    df_filtered = df.loc[start_date:end_date]
    return df_filtered


def apply_background_spans(ax):
    # 2019-07-03 ~ 2019-07-04
    ax.axvspan(
        pd.Timestamp('2019-07-03'),
        pd.Timestamp('2019-07-04'),
        facecolor='none',
        edgecolor='#900000',
        hatch='//////',
        linewidth=0.0,
        zorder=0
    )

    # 2019-07-05 ~ 2019-07-06
    ax.axvspan(
        pd.Timestamp('2019-07-05'),
        pd.Timestamp('2019-07-06'),
        facecolor='none',
        edgecolor='black',
        hatch='//////',
        linewidth=0.0,
        zorder=0
    )


def plot_displacement_series(ax, df_filtered):
    color_de = '#0a9396'
    color_dn = '#bb3e03'

    # dE：
    ax.plot(
        df_filtered.index, df_filtered['dE'],
        label='dE',
        color=color_de,
        lw=1.6,
        marker='o',
        ms=5.6,
        mfc='white',      
        mec=color_de,
        mew=1.0,
        zorder=3
    )

    # dN：
    ax.plot(
        df_filtered.index, df_filtered['dN'],
        label='dN',
        color=color_dn,
        lw=1.6,
        marker='o',
        ms=5.6,
        mfc=color_dn,     
        mec=color_dn,
        mew=0.8,
        zorder=3
    )


def format_xaxis(ax):
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d'))
    ax.xaxis.set_minor_locator(mdates.DayLocator(interval=1))

    ax.tick_params(axis='x', which='major', direction='in', length=4, width=0.8, pad=4)
    ax.tick_params(axis='x', which='minor', direction='in', length=2.2, width=0.7)


def format_yaxis(ax, df_filtered):
    y_min = min(df_filtered[['dE', 'dN']].min().min(), 0)
    y_max = max(df_filtered[['dE', 'dN']].max().max(), 0)
    y_range = y_max - y_min

    if y_range > 0:
        tick_interval = math.ceil(y_range / 4 * 10) / 10

        ax.set_ylim(round(y_min - 0.03, 2), round(y_max + 0.03, 2))
        ax.yaxis.set_major_locator(MultipleLocator(tick_interval))
        ax.yaxis.set_minor_locator(MultipleLocator(tick_interval / 5))
        ax.tick_params(axis='y', which='major', direction='in', length=4, width=0.8, pad=4)
        ax.tick_params(axis='y', which='minor', direction='in', length=2.2, width=0.7)


def style_axes(ax):
    ax.set_xlabel('Date (2019-07)', labelpad=5)
    ax.set_ylabel('Displacement (m)', labelpad=6)

    for spine in ax.spines.values():
        spine.set_linewidth(0.9)
        spine.set_color('black')

    ax.grid(False)

    ax.legend(
        loc='upper left',
        frameon=False,
        handlelength=2.0,
        borderaxespad=0.8
    )


def save_figure(fig, png_path, pdf_path):
    fig.savefig(png_path, dpi=600, bbox_inches='tight')
    fig.savefig(pdf_path, bbox_inches='tight')
    print("Successfully saved figure!")


def main():
    csv_path = 'D://a//master//Earthquake-US//Data//20190706-102//PosData-7//P595.csv'
    start_date = '2019-07-01'
    end_date = '2019-07-08'

    png_path = 'D:/a/master/Earthquake-US/Fig-Use-2/Fig-1/P595_Use_nature.png'
    pdf_path = 'D:/a/master/Earthquake-US/Fig-Use-2/Fig-1/P595_Use_nature.pdf'

    set_plot_style()
    df_filtered = load_and_filter_data(csv_path, start_date, end_date)

    fig, ax = plt.subplots(figsize=(3.54, 2.6), dpi=300)

    apply_background_spans(ax)
    plot_displacement_series(ax, df_filtered)
    format_xaxis(ax)
    format_yaxis(ax, df_filtered)
    style_axes(ax)

    plt.tight_layout(pad=0.5)
    save_figure(fig, png_path, pdf_path)
    plt.close(fig)


if __name__ == '__main__':
    main()
'''

Fig-2

In [ ]:
#The data of W, S, dS, and Phi
'''
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import glob
import os


# ========================
# 1. Data Loading and Preprocessing
# ========================
def load_and_merge_data(csv_dir):
    """Load and merge all CSV files"""
    files = glob.glob(f"{csv_dir}/*.csv")
    dfs = []
    for file in files:
        df = pd.read_csv(file)
        df['site_id'] = file.split('/')[-1].split('.')[0]  # Use the filename as the site ID
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)


def preprocess_gnss_data(df, time_step=1):
    """Preprocess GNSS data (only two displacement components are processed)"""
    # Convert the time column
    df['time'] = pd.to_datetime(df['YYYYMMDD'], format='%Y%m%d')

    # Downsample according to the time step
    sampled_times = df['time'].unique()[::time_step]
    df_sampled = df[df['time'].isin(sampled_times)].copy()

    # Standardize the displacement data for each site
    for component in ['dE', 'dN']:
        df_sampled[f'{component}_standardized'] = 0
        for site in df_sampled['site_id'].unique():
            site_mask = df_sampled['site_id'] == site
            scaler = StandardScaler()
            df_sampled.loc[site_mask, f'{component}_standardized'] = scaler.fit_transform(
                df_sampled.loc[site_mask, component].values.reshape(-1, 1)).flatten()

    # Check data length consistency
    site_lengths = df_sampled.groupby('site_id').size()
    if len(site_lengths.unique()) > 1:
        print("Warning: Data lengths are inconsistent for some sites; automatic trimming will be performed")
        M = min(site_lengths)  # Take the minimum length
        valid_sites = site_lengths[site_lengths == M].index
        df_sampled = df_sampled[df_sampled['site_id'].isin(valid_sites)]
    else:
        M = site_lengths[0]

    # Construct the microstate matrix (M x 2N) - each site has 2 displacement variables (dU removed)
    sites = sorted(df_sampled['site_id'].unique())
    N_T = 2 * len(sites)  # Each site has 2 degrees of freedom (dE, dN)
    A = np.zeros((M, N_T))

    for i, site in enumerate(sites):
        site_data = df_sampled[df_sampled['site_id'] == site].iloc[:M]
        # Standardized displacement values (dE and dN only)
        A[:, 2 * i] = site_data['dE_standardized'].values
        A[:, 2 * i + 1] = site_data['dN_standardized'].values

    # Normalization
    C_0 = np.sum(A ** 2)
    A_normalized = A / np.sqrt(C_0)

    return A_normalized, sites, sampled_times[:M], df_sampled


# ========================
# 2. Eigenmicrostate Calculation
# ========================
def compute_eigen_microstates(A):
    """Calculate eigenmicrostates using SVD"""
    U, S, Vt = np.linalg.svd(A.T, full_matrices=False)
    eigenvalues = S ** 2
    eigenmicrostates = U  # Each column is an eigenmicrostate
    return eigenvalues, eigenmicrostates, Vt.T, S


# ========================
# 3. Regional Division and Collective Motion Index Calculation
# ========================
def divide_into_regions(df, center_lat=35.7695, center_lon=242.4006667, delta=1):
    """Divide sites into a 3x3 grid of regions"""
    lat_delta = 1.07 / 3  # Divide the latitude direction into 3 rows
    lon_delta = 1.47 / 3  # Divide the longitude direction into 3 columns

    # Initialize the region dictionary
    regions = {}

    # Create a 3x3 grid
    for i in range(3):
        for j in range(3):
            # Calculate the latitude and longitude ranges of the current region
            lat_low = center_lat + (i - 1.5) * lat_delta
            lat_high = center_lat + (i - 1.5) * lat_delta + lat_delta
            lon_low = center_lon + (j - 1.5) * lon_delta
            lon_high = center_lon + (j - 1.5) * lon_delta + lon_delta


            # Select sites within the current region
            region_mask = (df['NLat'] >= lat_low) & (df['NLat'] < lat_high) & \
                          (df['Elong'] >= lon_low) & (df['Elong'] < lon_high)
            region_sites = df[region_mask]['site_id'].unique()

            regions[f'region_{i}_{j}'] = {
                'sites': region_sites,
                'lat_range': (lat_low, lat_high),
                'lon_range': (lon_low, lon_high),
                'center': (center_lat + (i - 1) * delta, center_lon + (j - 1) * delta)
            }

    return regions


def calculate_entropy(eigenvalues):
    """Calculate entropy"""
    # Unnormalized eigenvalues
    lambda_sq = eigenvalues
    # Calculate entropy
    S = -np.sum(lambda_sq * np.log(lambda_sq))
    return S


def calculate_entropy_change_rate(entropy_list):
    """Calculate the entropy change rate ΔS"""
    delta_S = []
    n = len(entropy_list)

    if n == 0:
        return delta_S

    # Process the first point (using forward difference)
    if n >= 2:
        delta_S.append(entropy_list[1] - entropy_list[0])
    else:
        delta_S.append(0)  # The change rate is 0 when there is only one point

    # Process intermediate points (using central difference)
    for i in range(1, n - 1):
        delta_S.append((entropy_list[i + 1] - entropy_list[i - 1]) / 2)

    # Process the last point (using backward difference)
    if n >= 2:
        delta_S.append(entropy_list[-1] - entropy_list[-2])

    return delta_S


def calculate_regional_phi(eigenmicrostate, regions, df_ref):
    """Calculate the collective motion index for each region"""
    phi_values = {}
    angle_values = {}
    site_phis = {}
    site_angles = {}

    # Variables used for global calculation
    all_sin_thetas = []
    all_cos_thetas = []
    regional_phis = []  # Store phi values for all regions

    for region_name, region_data in regions.items():
        sites_in_region = region_data['sites']
        if len(sites_in_region) == 0:
            phi_values[region_name] = 0.0
            angle_values[region_name] = 0.0
            site_phis[region_name] = []
            site_angles[region_name] = []
            continue

        # Obtain the indices of these sites in the eigenmicrostate
        all_sites = sorted(df_ref['site_id'].unique())
        indices = []
        for site in sites_in_region:
            site_idx = all_sites.index(site)
            # Each site has 2 components (dE, dN)
            indices.extend([2 * site_idx, 2 * site_idx + 1])

        # Extract the eigenmicrostate components for this region
        region_components = eigenmicrostate[indices]

        # Calculate theta (angle) and phi for each site
        dE = region_components[::2]  # Eastward displacement component
        dN = region_components[1::2]  # Northward displacement component
        thetas = np.arctan2(dE, dN)  # Calculate angles using atan2

        # Collect all angles for global calculation
        all_sin_thetas.extend(np.sin(thetas))
        all_cos_thetas.extend(np.cos(thetas))

        # Calculate phi for each site
        site_phi = np.sqrt(dE ** 2 + dN ** 2)

        # Calculate A and B according to the phi formula
        A = np.mean(np.sin(thetas))
        B = np.mean(np.cos(thetas))

        # Calculate phi and the mean angle
        phi = A ** 2 + B ** 2
        angle = np.arctan2(A, B)  # Mean angle

        phi_values[region_name] = phi
        angle_values[region_name] = angle
        site_phis[region_name] = site_phi
        site_angles[region_name] = thetas
        regional_phis.append(phi)  # Collect regional phi values

    # Calculate global phi and global angle
    if all_sin_thetas and all_cos_thetas:
        A_global = np.mean(all_sin_thetas)
        B_global = np.mean(all_cos_thetas)
        global_phi = A_global ** 2 + B_global ** 2
        global_angle = np.arctan2(A_global, B_global)
    else:
        global_phi = 0.0
        global_angle = 0.0

    # Calculate the average phi value (phi values from nine regions divided by 8)
    average_phi = sum(regional_phis) / 9 if len(regional_phis) > 0 else 0.0

    return phi_values, angle_values, global_phi, average_phi, site_phis, site_angles


# ========================
# 5. Main Analysis Workflow - Output Data Only, No Plotting
# ========================
def analyze_temporal_evolution(df, earthquake_df, window_size=30, step=1, output_dir=None):
    """Analyze temporal evolution - output data only, no plotting"""
    all_times = pd.to_datetime(df['YYYYMMDD'], format='%Y%m%d').unique()
    eigenvalues_list = []
    entropy_list = []
    entropy_change_rates = []
    global_phi_list = []
    average_phi_list = []
    diff_phi_list = []  # Added: difference list
    window_center_dates = []

    # Divide regions
    regions = divide_into_regions(df)

    for start in range(0, len(all_times) - window_size, step):
        window_times = all_times[start:start + window_size]
        window_end = window_times[-1]
        window_center_dates.append(window_end)

        df_window = df[df['time'].isin(window_times)]
        A, sites, _, df_sampled = preprocess_gnss_data(df_window)
        eigenvalues, eigenmicrostates, Vt, S = compute_eigen_microstates(A)

        # Store eigenvalues and calculate entropy
        eigenvalues_list.append(eigenvalues[:5])
        entropy_list.append(calculate_entropy(eigenvalues))

        # Calculate and store the global Phi value and average phi value
        _, _, global_phi, average_phi, _, _ = calculate_regional_phi(eigenmicrostates[:, 0], regions, df_sampled)
        global_phi_list.append(global_phi)
        average_phi_list.append(average_phi)
        diff_phi_list.append(average_phi - global_phi)

    # Calculate the entropy change rate
    entropy_change_rates = calculate_entropy_change_rate(entropy_list)

    # ========================
    # Output Data to CSV Files
    # ========================

    # 1. Output eigenvalue, entropy, and entropy change rate data
    eigenvalues_data = []
    for i, date in enumerate(window_center_dates):
        row = {
            'date': date.strftime('%Y-%m-%d'),
            'lambda_1': eigenvalues_list[i][0] if len(eigenvalues_list[i]) > 0 else np.nan,
            'lambda_2': eigenvalues_list[i][1] if len(eigenvalues_list[i]) > 1 else np.nan,
            'lambda_3': eigenvalues_list[i][2] if len(eigenvalues_list[i]) > 2 else np.nan,
            'lambda_4': eigenvalues_list[i][3] if len(eigenvalues_list[i]) > 3 else np.nan,
            'lambda_5': eigenvalues_list[i][4] if len(eigenvalues_list[i]) > 4 else np.nan,
            'entropy': entropy_list[i],
            'entropy_change_rate': entropy_change_rates[i] if i < len(entropy_change_rates) else np.nan
        }
        eigenvalues_data.append(row)

    eigenvalues_df = pd.DataFrame(eigenvalues_data)

    # 2. Output Phi value data
    phi_data = []
    for i, date in enumerate(window_center_dates):
        row = {
            'date': date.strftime('%Y-%m-%d'),
            'global_phi': global_phi_list[i],
            'average_phi': average_phi_list[i],
            'diff_phi': diff_phi_list[i]  # Difference
        }
        phi_data.append(row)

    phi_df = pd.DataFrame(phi_data)

    # 3. Save to CSV files
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

        # Save eigenvalue data
        eigenvalues_file = os.path.join(output_dir, 'eigenvalues_entropy_data.csv')
        eigenvalues_df.to_csv(eigenvalues_file, index=False, encoding='utf-8-sig')
        print(f"Eigenvalue, entropy, and entropy change rate data saved to: {eigenvalues_file}")

        # Save Phi value data
        phi_file = os.path.join(output_dir, 'phi_values_data.csv')
        phi_df.to_csv(phi_file, index=False, encoding='utf-8-sig')
        print(f"Phi value data saved to: {phi_file}")

        # Print data statistics
        print("\n=== Data Statistics ===")
        print(f"Total number of time windows: {len(window_center_dates)}")
        print(
            f"Time range: {window_center_dates[0].strftime('%Y-%m-%d')} to {window_center_dates[-1].strftime('%Y-%m-%d')}")
        print(f"\nEigenvalue Statistics:")
        print(f"Mean λ₁: {np.mean([eig[0] for eig in eigenvalues_list]):.6f}")
        print(f"Standard deviation of λ₁: {np.std([eig[0] for eig in eigenvalues_list]):.6f}")
        print(f"\nEntropy Statistics:")
        print(f"Mean entropy: {np.mean(entropy_list):.6f}")
        print(f"Standard deviation of entropy: {np.std(entropy_list):.6f}")
        print(f"\nPhi Value Statistics:")
        print(f"Mean global Phi: {np.mean(global_phi_list):.6f}")
        print(f"Mean average Phi: {np.mean(average_phi_list):.6f}")
        print(f"Mean difference: {np.mean(diff_phi_list):.6f}")

    # Return data for further analysis
    return {
        'window_center_dates': window_center_dates,
        'eigenvalues_list': eigenvalues_list,
        'entropy_list': entropy_list,
        'entropy_change_rates': entropy_change_rates,
        'global_phi_list': global_phi_list,
        'average_phi_list': average_phi_list,
        'diff_phi_list': diff_phi_list,
        'eigenvalues_df': eigenvalues_df,
        'phi_df': phi_df
    }


# ========================
# Main Program Workflow
# ========================
if __name__ == "__main__":
    # Set output directory
    output_dir = "D://a//master//Earthquake-US//Progect//Output//20190706-20//IMS-Value"

    # 1. Load data
    csv_dir = "D://a//master//Earthquake-US//Data//20190706-20//PosData-7"
    df_merged = load_and_merge_data(csv_dir)
    earthquake_df = pd.read_csv(
        "D://a//master//Earthquake-US//Progect//Output//20190706-20//QueryData//daily_max_mag_earthquakes.csv")

    # 2. Preprocessing (horizontal displacement data only)
    A, sites, times, df_sampled = preprocess_gnss_data(df_merged)

    # 3. Analyze temporal evolution (output data only, no plotting)
    results = analyze_temporal_evolution(df_merged, earthquake_df, output_dir=output_dir)
'''

In [ ]:
#The data of E
'''
import os
import re
import numpy as np
import pandas as pd

# ============================================================
# Eigenmicrostate statistical mechanics: E, U, F calculation
# Based on:
#   E_I = ln(P_1 / P_I)
#   F_EM = ln(P_1)
#   F_EM = U_EM - S_EM  ->  U_EM = F_EM + S_EM
# Here the modal weights W^I are treated as probabilities P_I.
# ============================================================

# ---------- Input ----------
input_file = r"D:\a\master\Earthquake-US\Fig-Over-Output\Fig-2\IMS_eigenvalues_entropy_data.xlsx"
sheet_name = "Sheet1"

# ---------- Output ----------
output_file = r"D:\a\master\Earthquake-US\Fig-Over-Output\Fig-2\IMS_E_U_F_results.csv"

# Column names in the supplied workbook
w1_col = "lambda_1_102"      # W^1 = P_1
entropy_col = "entropy_102"  # S_EM calculated from the full eigenvalue spectrum


def natural_mode_key(col_name):
    """Sort lambda_1, lambda_2, ... in numerical order."""
    m = re.search(r"lambda_(\d+)", col_name)
    return int(m.group(1)) if m else 10**9


def main():
    # 1. Read data
    df = pd.read_excel(input_file, sheet_name=sheet_name)

    if w1_col not in df.columns:
        raise KeyError(f"Missing first-mode column: {w1_col}")
    if entropy_col not in df.columns:
        raise KeyError(f"Missing entropy column: {entropy_col}")

    # Detect all eigenvalue/weight columns available in the file
    weight_cols = sorted(
        [c for c in df.columns if re.fullmatch(r"lambda_\d+_102", str(c))],
        key=natural_mode_key,
    )

    if not weight_cols:
        raise ValueError("No eigenvalue columns such as lambda_1_102 were found.")

    # 2. W^1 = P_1
    W1 = pd.to_numeric(df[w1_col], errors="coerce")
    S = pd.to_numeric(df[entropy_col], errors="coerce")

    # Logarithms require positive probabilities
    if (W1 <= 0).any():
        bad_rows = df.index[W1 <= 0].tolist()
        raise ValueError(f"W^1 contains zero/negative values at rows: {bad_rows[:10]}")

    result = pd.DataFrame()

    # Preserve date if present
    if "date" in df.columns:
        result["date"] = pd.to_datetime(df["date"], errors="coerce").dt.strftime("%Y-%m-%d")

    result["W1"] = W1
    result["S_EM"] = S

    # 3. Partition function and free energy
    # Z_EM = 1 / P_1
    # F_EM = -ln(Z_EM) = ln(P_1)
    result["Z_EM"] = 1.0 / W1
    result["F_EM"] = np.log(W1)

    # 4. Energy spectrum
    # E_I = ln(P_1 / P_I)
    # Therefore E_1 = 0 by construction.
    energy_cols = []
    for col in weight_cols:
        mode = natural_mode_key(col)
        Wi = pd.to_numeric(df[col], errors="coerce")

        # For zero/negative Wi, E_I is undefined/infinite; use NaN in CSV.
        valid = Wi > 0
        E = pd.Series(np.nan, index=df.index, dtype=float)
        E.loc[valid] = np.log(W1.loc[valid] / Wi.loc[valid])

        out_col = f"E{mode}"
        result[out_col] = E
        energy_cols.append(out_col)

    # 5. Full internal energy
    # Article gives F_EM = U_EM - S_EM, hence U_EM = F_EM + S_EM.
    # This is preferable here because the supplied file stores only the first
    # few W^I columns, while entropy_102 was calculated from the full spectrum.
    result["U_EM"] = result["F_EM"] + result["S_EM"]

    # Arrange columns
    first_cols = [c for c in ["date", "W1", "S_EM", "Z_EM", "F_EM"] if c in result.columns]
    result = result[first_cols + energy_cols + ["U_EM"]]

    # 6. Output CSV
    result.to_csv(output_file, index=False, encoding="utf-8-sig", float_format="%.10f")

    print("Calculation completed.")
    print(f"Input : {os.path.abspath(input_file)}")
    print(f"Output: {os.path.abspath(output_file)}")
    print("\nFirst 5 rows:")
    print(result.head().to_string(index=False))


if __name__ == "__main__":
    main()
'''

In [ ]:
# First mode, second eigenmicrostate energy, entropy, and entropy change
# Plotting data time range: 2018-09-06—2020-05-06
'''
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import AutoMinorLocator
from matplotlib.lines import Line2D


# ========================
# Global plotting parameters: publication style
# ========================
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['mathtext.fontset'] = 'stix'
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['xtick.labelsize'] = 11
plt.rcParams['ytick.labelsize'] = 11
plt.rcParams['legend.fontsize'] = 11
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['axes.linewidth'] = 0.7
plt.rcParams['xtick.major.width'] = 0.7
plt.rcParams['ytick.major.width'] = 0.7
plt.rcParams['xtick.minor.width'] = 0.5
plt.rcParams['ytick.minor.width'] = 0.5


def load_data(excel_path, csv_path):
    # Excel: W1, S, ΔS
    df_ws = pd.read_excel(excel_path)
    df_ws['date'] = pd.to_datetime(df_ws['date'])

    required_excel = [
        'date',
        'lambda_1_102',
        'entropy_102',
        'entropy_change_rate_102'
    ]
    for col in required_excel:
        if col not in df_ws.columns:
            raise ValueError(f"Required column missing from Excel file: {col}")

    # CSV: E2
    df_e = pd.read_csv(csv_path)
    df_e['date'] = pd.to_datetime(df_e['date'])

    required_csv = ['date', 'E2']
    for col in required_csv:
        if col not in df_e.columns:
            raise ValueError(f"Required column missing from CSV file: {col}")

    # Merge by date
    df = pd.merge(
        df_ws[required_excel],
        df_e[required_csv],
        on='date',
        how='inner'
    ).sort_values('date').reset_index(drop=True)

    if df.empty:
        raise ValueError("No common data remain after merging Excel and CSV by date. Please check the date format.")

    return df


def draw_segment_mean(ax,mean_value, start_date, end_date, color='#777777', linestyle=':', linewidth=0.8, zorder=2):
    ax.hlines(
        y=mean_value,
        xmin=start_date,
        xmax=end_date,
        color=color,
        linestyle=linestyle,
        linewidth=linewidth,
        zorder=zorder
    )


def style_right_axis(ax):
    ax.set_facecolor('none')
    ax.grid(False)
    ax.yaxis.set_minor_locator(AutoMinorLocator(4))

    for spine in ax.spines.values():
        spine.set_linewidth(0.7)
        spine.set_color('0.2')

    ax.tick_params(
        axis='y',
        which='major',
        direction='in',
        left=False,
        right=True,
        labelright=True,
        length=4,
        width=0.7,
        colors='black'
    )
    ax.tick_params(
        axis='y',
        which='minor',
        direction='in',
        left=False,
        right=True,
        length=2.5,
        width=0.5,
        colors='black'
    )


def plot_two_rows_one_column(df, output_dir=None):

    event_date = pd.to_datetime('2019-07-06')

    plot_start = pd.to_datetime('2018-09-06')
    plot_end = pd.to_datetime('2020-05-06')

    mean_period1_start = pd.to_datetime('2018-09-06')
    mean_period1_end = pd.to_datetime('2019-07-05')

    mean_period2_start = pd.to_datetime('2019-08-07')
    mean_period2_end = pd.to_datetime('2020-05-06')

    df_plot = df[
        (df['date'] >= plot_start) &
        (df['date'] <= plot_end)
    ].copy()

    if df_plot.empty:
        raise ValueError(
            f"No data are available within the specified time range {plot_start.date()} to {plot_end.date()}."
        )

    df_period1 = df_plot[
        (df_plot['date'] >= mean_period1_start) &
        (df_plot['date'] <= mean_period1_end)
    ].copy()

    df_period2 = df_plot[
        (df_plot['date'] >= mean_period2_start) &
        (df_plot['date'] <= mean_period2_end)
    ].copy()

    if df_period1.empty or df_period2.empty:
        raise ValueError("No data are available within the mean calculation intervals. Please check the date range.")

    # ========================
    # Advanced academic color scheme: low saturation and balanced cool/warm tones
    # ========================
    color_w1 = '#1f4e79'   # Dark blue
    color_e2 = '#C88732'   # Golden yellow / ochre
    color_s = '#2e8b57'    # Low-saturation dark green
    color_ds = '#c23b22'   # Low-saturation brick red
    
    quake_color = '#202020'
    mean_color = '#777777'

    # Mean values
    mean_w1_p1 = df_period1['lambda_1_102'].mean()
    mean_e2_p1 = df_period1['E2'].mean()
    mean_s_p1 = df_period1['entropy_102'].mean()
    mean_ds_p1 = df_period1['entropy_change_rate_102'].mean()

    mean_w1_p2 = df_period2['lambda_1_102'].mean()
    mean_e2_p2 = df_period2['E2'].mean()
    mean_s_p2 = df_period2['entropy_102'].mean()
    mean_ds_p2 = df_period2['entropy_change_rate_102'].mean()

    major_ticks = pd.to_datetime([
        '2018-11-01',
        '2019-03-01',
        '2019-07-01',
        '2019-11-01',
        '2020-03-01'
    ])

    # ========================
    # Two rows and one column
    # ========================
    fig, axes = plt.subplots(
        2, 1,
        figsize=(4.4, 4.4),
        sharex=True,
        facecolor='white'
    )

    ax_top = axes[0]
    ax_bottom = axes[1]

    for ax in axes:
        ax.set_facecolor('white')
        ax.grid(False, which='both', axis='both')
        ax.set_xlim(plot_start, plot_end)

        ax.set_xticks(major_ticks)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=1))
        ax.yaxis.set_minor_locator(AutoMinorLocator(5))

        ax.axvline(
            event_date,
            color=quake_color,
            linestyle='--',
            linewidth=0.9,
            zorder=1
        )

        for spine in ax.spines.values():
            spine.set_linewidth(0.7)
            spine.set_color('0.2')

        ax.tick_params(
            axis='x',
            which='major',
            direction='in',
            bottom=True,
            top=False,
            length=4,
            width=0.7
        )
        ax.tick_params(
            axis='x',
            which='minor',
            direction='in',
            bottom=True,
            top=False,
            length=2.5,
            width=0.5
        )
        ax.tick_params(
            axis='y',
            which='major',
            direction='in',
            left=True,
            right=False,
            length=4,
            width=0.7
        )
        ax.tick_params(
            axis='y',
            which='minor',
            direction='in',
            left=True,
            right=False,
            length=2.5,
            width=0.5
        )

    ax_top.tick_params(labelbottom=False)
    ax_top.set_xlabel('')
    ax_bottom.set_xlabel('Date')

    main_quake_handle = Line2D(
        [0], [0],
        color=quake_color,
        linestyle='--',
        linewidth=0.9,
        label='Main'
    )

    # ========================
    # Top panel: left axis W1
    # ========================
    line_w1, = ax_top.plot(
        df_plot['date'],
        df_plot['lambda_1_102'],
        color=color_w1,
        linewidth=1.45,
        marker='o',
        markersize=1.6,
        markevery=10,
        zorder=3,
        label=r'$W^1$'
    )

    draw_segment_mean(
        ax_top, mean_w1_p1,
        mean_period1_start, mean_period1_end,
        color=mean_color
    )
    draw_segment_mean(
        ax_top, mean_w1_p2,
        mean_period2_start, mean_period2_end,
        color=mean_color
    )

    ax_top.set_ylabel(r'$W^1$', color='black')
    ax_top.tick_params(axis='y', labelcolor='black')

    # ========================
    # Top panel: right axis E2
    # ========================
    ax_e2 = ax_top.twinx()
    style_right_axis(ax_e2)

    line_e2, = ax_e2.plot(
        df_plot['date'],
        df_plot['E2'],
        color=color_e2,
        linewidth=1.35,
        marker='o',
        markersize=1.6,
        markevery=10,
        zorder=3,
        label=r'$E_2$'
    )

    # draw_segment_mean(
    #     ax_e2, mean_e2_p1,
    #     mean_period1_start, mean_period1_end,
    #     color=mean_color
    # )
    # draw_segment_mean(
    #     ax_e2, mean_e2_p2,
    #     mean_period2_start, mean_period2_end,
    #     color=mean_color
    # )
    
    ax_e2.set_yticks([0.6, 1.2, 1.8])
    ax_e2.set_ylabel(r'$E_2$', color='black', labelpad=3)

    ax_top.legend(
        handles=[line_w1, line_e2, main_quake_handle],
        loc='upper left',
        frameon=False,
        handlelength=1.8,
        borderpad=0.2
    )

    # ========================
    # Bottom panel: left axis S
    # ========================
    ax_bottom.set_yticks([2.0, 2.4, 2.8])

    line_s, = ax_bottom.plot(
        df_plot['date'],
        df_plot['entropy_102'],
        color=color_s,
        linewidth=1.45,
        marker='o',
        markersize=1.6,
        markevery=10,
        zorder=3,
        label=r'$S$'
    )

    draw_segment_mean(
        ax_bottom, mean_s_p1,
        mean_period1_start, mean_period1_end,
        color=mean_color
    )
    draw_segment_mean(
        ax_bottom, mean_s_p2,
        mean_period2_start, mean_period2_end,
        color=mean_color
    )

    ax_bottom.set_ylabel(r'$S$', color='black')
    ax_bottom.tick_params(axis='y', labelcolor='black')

    # ========================
    # Bottom panel: right axis ΔS
    # ========================
    ax_ds = ax_bottom.twinx()
    style_right_axis(ax_ds)

    line_ds, = ax_ds.plot(
        df_plot['date'],
        df_plot['entropy_change_rate_102'],
        color=color_ds,
        linewidth=1.30,
        marker='o',
        markersize=1.6,
        markevery=10,
        zorder=3,
        label=r'$\Delta S$'
    )

    # draw_segment_mean(
    #     ax_ds, mean_ds_p1,
    #     mean_period1_start, mean_period1_end,
    #     color=mean_color
    # )
    # draw_segment_mean(
    #     ax_ds, mean_ds_p2,
    #     mean_period2_start, mean_period2_end,
    #     color=mean_color
    # )

    ax_ds.set_yticks([-0.2, 0.0, 0.2])
    ax_ds.set_ylabel(r'$\Delta S$', color='black', labelpad=3)

    ax_bottom.legend(
        handles=[line_s, line_ds],
        loc='lower left',
        frameon=False,
        handlelength=1.8,
        borderpad=0.2
    )

    # ========================
    # Layout
    # ========================
    plt.subplots_adjust(
        left=0.15,
        right=0.85,
        top=0.97,
        bottom=0.14,
        hspace=0.06
    )

    # ========================
    # Save figures
    # ========================
    if output_dir is not None:
        os.makedirs(output_dir, exist_ok=True)

        png_path = os.path.join(
            output_dir,
            'W1_E2_Entropy_DeltaS_2x1_blue_yellow_green_red.png'
        )
        pdf_path = os.path.join(
            output_dir,
            'W1_E2_Entropy_DeltaS_2x1_blue_yellow_green_red.pdf'
        )

        plt.savefig(
            png_path,
            dpi=600,
            bbox_inches='tight',
            facecolor='white'
        )
        plt.savefig(
            pdf_path,
            format='pdf',
            bbox_inches='tight',
            facecolor='white'
        )

        print(f'\nFigure saved as:\n{png_path}\n{pdf_path}')

    plt.close()


if __name__ == '__main__':

    excel_path = (
        r'D:\a\master\Earthquake-US'
        r'\Fig-Use-2\Fig-2'
        r'\IMS_eigenvalues_entropy_data.xlsx'
    )

    csv_path = (
        r'D:\a\master\Earthquake-US'
        r'\Fig-Over-Output\Fig-2'
        r'\IMS_E_U_F_results.csv'
    )

    output_dir = (
        r'D:\a\master\Earthquake-US'
        r'\Fig-Over-Output\Fig-2'
    )

    df = load_data(excel_path, csv_path)

    plot_two_rows_one_column(
        df,
        output_dir=output_dir
    )
'''

Fig-3

In [ ]:
#The sources of the data are the same as those of 'The data of W, S, dS, and Phi'.
# (a) Temporal evolution of the three phi values
'''
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import AutoMinorLocator, FixedLocator
from matplotlib.patches import Rectangle

# ========================
# 0. Global plotting parameters (Nature double-column style)
# ========================
FIG_WIDTH = 8.8
FIG_HEIGHT = 3.0
# Global text color
plt.rcParams['text.color'] = 'black'
plt.rcParams['axes.labelcolor'] = 'black'
plt.rcParams['xtick.color'] = 'black'
plt.rcParams['ytick.color'] = 'black'

plt.rcParams['font.family'] = 'Arial'
plt.rcParams['mathtext.fontset'] = 'stix'
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 11
plt.rcParams['ytick.labelsize'] = 11
plt.rcParams['legend.fontsize'] = 12

plt.rcParams['axes.linewidth'] = 0.7
plt.rcParams['xtick.major.width'] = 0.7
plt.rcParams['ytick.major.width'] = 0.7
plt.rcParams['xtick.minor.width'] = 0.5
plt.rcParams['ytick.minor.width'] = 0.5

plt.rcParams['xtick.major.size'] = 3.5
plt.rcParams['ytick.major.size'] = 3.5
plt.rcParams['xtick.minor.size'] = 2.0
plt.rcParams['ytick.minor.size'] = 2.0

plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['savefig.facecolor'] = 'white'
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'


# ========================
# 1. Read Excel data
# ========================
def load_phi_data(excel_path):
    df = pd.read_excel(excel_path)

    df['date'] = pd.to_datetime(df['date'])
    df['global_phi'] = pd.to_numeric(df['global_phi_102'], errors='coerce')
    df['average_phi'] = pd.to_numeric(df['average_phi_102'], errors='coerce')
    df['diff_phi'] = pd.to_numeric(df['diff_phi_102'], errors='coerce')

    df = df[['date', 'global_phi', 'average_phi', 'diff_phi']].dropna()
    df = df.sort_values('date').reset_index(drop=True)

    return df


# ========================
# 2. Draw dashed boxes
# ========================
def add_custom_box(ax, df, start_date, end_date, y_bottom=None, y_top=None, line_cols=('global_phi', 'average_phi', 'diff_phi'),
                   auto_pad_ratio=0.08, auto_min_pad=0.03, edgecolor='#99582a', linewidth=1.3, linestyle=(0, (7, 3.5)) ):
    """
    Draw a box for a time interval:
    1) If both y_bottom and y_top are provided, use the manually specified values
    2) If not provided, automatically determine the range based on the curves within the interval
    """
    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)

    x0 = mdates.date2num(start_date)
    x1 = mdates.date2num(end_date)

    # ---------- Manual mode ----------
    if (y_bottom is not None) and (y_top is not None):
        y0 = y_bottom
        y1 = y_top

    # ---------- Automatic mode ----------
    else:
        sub = df[(df['date'] >= start_date) & (df['date'] <= end_date)].copy()
        if sub.empty:
            return

        vals = sub[list(line_cols)].values.flatten()
        vals = pd.Series(vals).dropna().values
        if len(vals) == 0:
            return

        y_min_local = vals.min()
        y_max_local = vals.max()
        y_range_local = y_max_local - y_min_local

        pad = max(y_range_local * auto_pad_ratio, auto_min_pad)

        y0 = y_min_local - pad
        y1 = y_max_local + pad

    rect = Rectangle(
        (x0, y0),
        x1 - x0,
        y1 - y0,
        fill=False,
        edgecolor=edgecolor,
        linewidth=linewidth,
        linestyle=linestyle,
        zorder=2
    )
    ax.add_patch(rect)


def highlight_points(ax, df, target_dates, columns_colors_markers,
                     size=18, edgecolor='black', linewidth=0.6, zorder=6):
    for d in target_dates:
        d = pd.to_datetime(d)
        row = df[df['date'] == d]
        if row.empty:
            continue

        for col, color, marker in columns_colors_markers:
            y = row[col].values[0]
            ax.scatter(
                d, y,
                s=size,
                color=color,
                marker=marker,
                linewidth=0,
                zorder=zorder
            )


# ========================
# 3. Plotting
# ========================
def plot_phi_nature_double_column(df, output_dir):
    # ========================
    # Plotting data time range: 2018-09-06 to 2020-05-06
    # ========================
    plot_start = pd.to_datetime('2018-09-06')
    plot_end = pd.to_datetime('2020-05-06')

    df = df[
        (df['date'] >= plot_start) &
        (df['date'] <= plot_end)
    ].copy()

    if df.empty:
        raise ValueError(
            f"No data are available within the specified time range {plot_start.date()} to {plot_end.date()}. Please check the date column in the Excel file."
        )

    fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT))
    
    # ------------------------
    # Main curve colors for a high-impact journal style
    # ------------------------
    color_global = '#1f4e79'
    color_avg    = '#ca6702'
    color_diff   = '#2e8b57'

    box1_color = '#ffbe0b'
    box2_color = '#fb5607'
    box3_color = '#8338ec'
    box4_color = '#3a86ff'

    # Curves
    ax.plot(
        df['date'], df['global_phi'],
        color=color_global,
        linewidth=1.55,
        label=r'$\Phi_{\mathrm{global}}$',
        zorder=4
    )

    ax.plot(
        df['date'], df['average_phi'],
        color=color_avg,
        linewidth=1.50,
        label=r'$\overline{\Phi}_{\mathrm{regional}}$',
        zorder=4
    )

    ax.plot(
        df['date'], df['diff_phi'],
        color=color_diff,
        linewidth=1.40,
        label=r'$\Delta \Phi_{\mathrm{rg}}$',
        zorder=4
    )

    # Axis labels
    ax.set_xlabel('Date')
    ax.set_ylabel(r'$\Phi$')

    # X-axis ticks
    major_tick_dates = pd.to_datetime([
        '2018-11-01',
        '2019-03-01',
        '2019-07-01',
        '2019-11-01',
        '2020-03-01'
    ])

    ax.xaxis.set_major_locator(FixedLocator(mdates.date2num(major_tick_dates)))
    ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

    ax.set_xlim(plot_start, plot_end)

    # Y-axis minor ticks
    ax.yaxis.set_minor_locator(AutoMinorLocator(5))

    # Tick style
    ax.tick_params(axis='both', which='major',
                   direction='in', bottom=True, left=True, top=False, right=False)
    ax.tick_params(axis='both', which='minor',
                   direction='in', bottom=True, left=True, top=False, right=False)

    for label in ax.get_xticklabels():
        label.set_ha('center')

    # Borders on all sides
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.7)
        spine.set_color('black')

    ax.grid(False)
    ax.margins(x=0.01)

    # Calculate the y range only from the truncated data
    y_min = min(df['global_phi'].min(), df['average_phi'].min(), df['diff_phi'].min())
    y_max = max(df['global_phi'].max(), df['average_phi'].max(), df['diff_phi'].max())
    y_range = y_max - y_min
    ax.set_ylim(y_min - 0.06 * y_range, y_max + 0.06 * y_range)

    # Interval boxes
    add_custom_box(
        ax, df,
        '2019-05-03', '2019-06-30',
        y_bottom=-0.10, y_top=0.97,
        edgecolor=box1_color,
        linewidth=1.3,
        linestyle=(0, (7, 3.5))
    )

    add_custom_box(
        ax, df,
        '2019-07-03', '2019-08-05',
        y_bottom=-0.10, y_top=0.97,
        edgecolor=box2_color,
        linewidth=1.3,
        linestyle=(0, (7, 3.5))
    )

    add_custom_box(
        ax, df,
        '2019-08-09', '2019-10-31',
        y_bottom=-0.10, y_top=0.97,
        edgecolor=box3_color,
        linewidth=1.3,
        linestyle=(0, (7, 3.5))
    )

    add_custom_box(
        ax, df,
        '2019-05-25', '2019-06-18',
        y_bottom=-0.05, y_top=0.5,
        edgecolor=box4_color,
        linewidth=1.3,
        linestyle=(0, (7, 3.5))
    )

    # Legend
    ax.legend(
        loc='lower right',
        bbox_to_anchor=(1.01, 0.42),
        frameon=False,
        handlelength=1,
        handletextpad=0.45,
        borderpad=0.12,
        labelspacing=0.22
    )

    plt.tight_layout(pad=0.45)

    # Save
    os.makedirs(output_dir, exist_ok=True)

    png_path = os.path.join(output_dir, 'Global_Phi_Temporal_Evolution.png')
    pdf_path = os.path.join(output_dir, 'Global_Phi_Temporal_Evolution.pdf')

    plt.savefig(png_path, dpi=600, bbox_inches='tight')
    plt.savefig(pdf_path, bbox_inches='tight')
    plt.close()

    print(f"PNG saved to: {png_path}")
    print(f"PDF saved to: {pdf_path}")


# ========================
# 4. Main program
# ========================
if __name__ == "__main__":
    excel_path = r"D:\a\master\Earthquake-US\Fig-Use-2\Fig-3\IMS_phi_values_data.xlsx"
    output_dir = r"D:\a\master\Earthquake-US\Fig-Over-Output\Fig-3"

    df_phi = load_phi_data(excel_path)
    plot_phi_nature_double_column(df_phi, output_dir=output_dir)
'''

In [ ]:
#The data of U
'''
import os
import glob
import gc
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler


# ========================
# 1) Data loading
# ========================
def load_and_merge_data(csv_dir: str) -> pd.DataFrame:
    files = glob.glob(os.path.join(csv_dir, "*.csv"))
    if not files:
        raise FileNotFoundError(f"No CSV files found in directory: {csv_dir}")

    dfs = []
    for fp in files:
        df = pd.read_csv(fp)
        site_id = os.path.splitext(os.path.basename(fp))[0]
        df["site_id"] = site_id

        if "YYYYMMDD" not in df.columns:
            raise ValueError(f"{fp} is missing the YYYYMMDD column")

        df["time"] = pd.to_datetime(df["YYYYMMDD"], format="%Y%m%d")
        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)


# ========================
# 2) 3x3 regional division
# ========================
def divide_into_regions(df: pd.DataFrame, center_lat=35.7695, center_lon=242.4006667, delta=1.0):
    if "NLat" not in df.columns or "Elong" not in df.columns:
        raise ValueError("Data are missing the NLat / Elong columns, so regions cannot be divided")

    regions = {}
    for i in range(3):
        for j in range(3):
            lat_low = center_lat + (i - 1) * delta - delta / 2
            lat_high = center_lat + (i - 1) * delta + delta / 2
            lon_low = center_lon + (j - 1) * delta - delta / 2
            lon_high = center_lon + (j - 1) * delta + delta / 2

            region_mask = (
                (df["NLat"] >= lat_low) & (df["NLat"] < lat_high) &
                (df["Elong"] >= lon_low) & (df["Elong"] < lon_high)
            )
            region_sites = df.loc[region_mask, "site_id"].unique()

            regions[f"region_{i}_{j}"] = {
                "sites": region_sites,
                "lat_range": (lat_low, lat_high),
                "lon_range": (lon_low, lon_high),
                "center": (center_lat + (i - 1) * delta, center_lon + (j - 1) * delta),  # (lat, lon)
            }
    return regions


# ========================
# 3) Window preprocessing -> A matrix (M x 2N)
# ========================
def preprocess_gnss_window(df_window: pd.DataFrame, time_step=1):
    need_cols = ["time", "site_id", "dE", "dN", "NLat", "Elong"]
    for c in need_cols:
        if c not in df_window.columns:
            raise ValueError(f"Window data are missing column: {c}")

    dfw = df_window[need_cols].copy()

    all_times = np.array(sorted(dfw["time"].unique()))
    sampled_times = all_times[::time_step]
    dfw = dfw[dfw["time"].isin(sampled_times)].copy()

    counts = dfw.groupby("site_id").size()
    if counts.empty:
        raise ValueError("No data in the window")

    M = int(counts.min())
    valid_sites = counts[counts == M].index
    dfw = dfw[dfw["site_id"].isin(valid_sites)].copy()

    sites = sorted(dfw["site_id"].unique())
    N = len(sites)
    if N == 0:
        raise ValueError("No valid sites in the window")

    dfw["dE_std"] = 0.0
    dfw["dN_std"] = 0.0

    for s in sites:
        sub = dfw[dfw["site_id"] == s].sort_values("time").iloc[:M]

        scaler_e = StandardScaler()
        scaler_n = StandardScaler()

        e_std = scaler_e.fit_transform(sub["dE"].to_numpy().reshape(-1, 1)).ravel()
        n_std = scaler_n.fit_transform(sub["dN"].to_numpy().reshape(-1, 1)).ravel()

        dfw.loc[sub.index, "dE_std"] = e_std
        dfw.loc[sub.index, "dN_std"] = n_std

    A = np.zeros((M, 2 * N), dtype=np.float32)
    for i, s in enumerate(sites):
        sub = dfw[dfw["site_id"] == s].sort_values("time").iloc[:M]
        A[:, 2 * i] = sub["dE_std"].to_numpy(dtype=np.float32)
        A[:, 2 * i + 1] = sub["dN_std"].to_numpy(dtype=np.float32)

    C0 = float(np.sum(A.astype(np.float64) ** 2))
    if C0 > 0:
        A /= np.sqrt(C0).astype(np.float32)

    times_used = dfw[dfw["site_id"] == sites[0]].sort_values("time").iloc[:M]["time"].to_numpy()
    return A, sites, times_used, dfw


# ========================
# 4) SVD -> first spatial mode
# ========================
def compute_first_eigenmicrostate(A: np.ndarray):
    U, S, Vt = np.linalg.svd(A, full_matrices=False)
    eigenmicrostate_1 = Vt[0, :].astype(np.float32)
    eigenvalues = (S.astype(np.float64) ** 2)
    return eigenvalues, eigenmicrostate_1


# ========================
# 5) Phi/Theta (following the site order after preprocessing)
# ========================
def calculate_regional_phi(eigenmicrostate_2N: np.ndarray, regions: dict, site_order: list):
    site_to_idx = {s: i for i, s in enumerate(site_order)}

    phi_values, theta_values = {}, {}
    site_phis, site_thetas = {}, {}
    all_sin, all_cos, regional_phis = [], [], []

    for region_name, region_data in regions.items():
        sites_in_region = region_data["sites"]
        if len(sites_in_region) == 0:
            phi_values[region_name] = 0.0
            theta_values[region_name] = 0.0
            site_phis[region_name] = np.array([], dtype=np.float32)
            site_thetas[region_name] = np.array([], dtype=np.float32)
            continue

        dE_list, dN_list, valid_sites = [], [], []
        for s in sites_in_region:
            if s not in site_to_idx:
                continue
            k = site_to_idx[s]
            dE_list.append(eigenmicrostate_2N[2 * k])
            dN_list.append(eigenmicrostate_2N[2 * k + 1])
            valid_sites.append(s)

        if len(dE_list) == 0:
            phi_values[region_name] = 0.0
            theta_values[region_name] = 0.0
            site_phis[region_name] = np.array([], dtype=np.float32)
            site_thetas[region_name] = np.array([], dtype=np.float32)
            continue

        dE = np.array(dE_list, dtype=np.float32)
        dN = np.array(dN_list, dtype=np.float32)

        thetas = np.arctan2(dE, dN)
        all_sin.extend(np.sin(thetas))
        all_cos.extend(np.cos(thetas))

        sphi = np.sqrt(dE ** 2 + dN ** 2)

        A = float(np.mean(np.sin(thetas)))
        B = float(np.mean(np.cos(thetas)))
        phi = A * A + B * B
        theta = float(np.arctan2(A, B))

        phi_values[region_name] = phi
        theta_values[region_name] = theta
        site_phis[region_name] = sphi
        site_thetas[region_name] = thetas
        regional_phis.append(phi)

    if all_sin and all_cos:
        A_g = float(np.mean(all_sin))
        B_g = float(np.mean(all_cos))
        phi_global = A_g * A_g + B_g * B_g
    else:
        phi_global = 0.0

    phi_region_mean = (sum(regional_phis) / len(regional_phis)) if regional_phis else 0.0
    return phi_values, theta_values, phi_global, phi_region_mean, site_phis, site_thetas


# ========================
# 6) Save plotting data to CSV
# ========================
def save_plot_data_to_csv(df_all: pd.DataFrame, output_dir: str, window_size=30, step=1, time_step_in_window=1, start_win=0, end_win=None,
                          center_lat=35.7695, center_lon=242.4006667, delta=1.0 ):
    os.makedirs(output_dir, exist_ok=True)

    all_times = np.array(sorted(df_all["time"].unique()))
    if len(all_times) < window_size:
        raise ValueError(f"Insufficient time points: len(all_times)={len(all_times)} < window_size={window_size}")

    regions = divide_into_regions(df_all, center_lat=center_lat, center_lon=center_lon, delta=delta)

    total_windows = (len(all_times) - window_size) // step + 1

    if start_win is None:
        start_win = 0
    start_win = max(0, int(start_win))

    if end_win is None:
        end_win = total_windows - 1
    end_win = min(total_windows - 1, int(end_win))

    if start_win > end_win:
        raise ValueError(f"start_win({start_win}) > end_win({end_win})")

    print(f"Total windows: {total_windows} | Saving windows: {start_win} -> {end_win}")

    summary_rows = []
    site_rows = []
    region_rows = []

    for win_idx in range(start_win, end_win + 1):
        start = win_idx * step
        window_times = all_times[start:start + window_size]
        window_end = pd.to_datetime(window_times[-1])

        df_window = df_all[df_all["time"].isin(window_times)]

        try:
            A, sites, times_used, df_ref = preprocess_gnss_window(df_window, time_step=time_step_in_window)
            _, em1 = compute_first_eigenmicrostate(A)

            phi_values, theta_values, phi_global, phi_region_mean, site_phis, site_thetas = calculate_regional_phi(
                em1, regions, sites
            )

            window_id = window_end.strftime("%Y%m%d")

            # ---- summary ----
            summary_rows.append({
                "window_index": win_idx,
                "window_id": window_id,
                "window_end": window_end.strftime("%Y-%m-%d"),
                "window_start": pd.to_datetime(window_times[0]).strftime("%Y-%m-%d"),
                "phi_global": phi_global,
                "phi_region_mean": phi_region_mean,
                "n_sites": len(sites),
                "n_times": len(times_used),
            })

            # Site coordinates
            site_lon, site_lat = {}, {}
            for s in sites:
                row = df_ref[df_ref["site_id"] == s].iloc[0]
                site_lon[s] = float(row["Elong"])
                site_lat[s] = float(row["NLat"])

            # ---- Arrows for each site ----
            for region_name, region_data in regions.items():
                sreg = [s for s in list(region_data["sites"]) if s in site_lon]
                if not sreg:
                    continue

                thetas = site_thetas[region_name]
                phis = site_phis[region_name]
                min_len = min(len(sreg), len(thetas), len(phis))
                if min_len == 0:
                    continue

                sreg = sreg[:min_len]
                thetas = thetas[:min_len]
                phis = phis[:min_len]

                for k, s in enumerate(sreg):
                    lon = site_lon[s]
                    lat = site_lat[s]
                    theta = float(thetas[k])
                    phi_s = float(phis[k])

                    u = phi_s * np.sin(theta) * 0.2
                    v = phi_s * np.cos(theta) * 0.2

                    site_rows.append({
                        "window_index": win_idx,
                        "window_id": window_id,
                        "window_end": window_end.strftime("%Y-%m-%d"),
                        "site_id": s,
                        "region_name": region_name,
                        "lon": lon,
                        "lat": lat,
                        "phi_site": phi_s,
                        "theta_site_rad": theta,
                        "theta_site_deg": np.degrees(theta),
                        "u": u,
                        "v": v,
                    })

            # ---- Arrows and text for each region ----
            for i in range(3):
                for j in range(3):
                    region_name = f"region_{i}_{j}"
                    r = regions[region_name]

                    sites_in_region = [s for s in r["sites"] if s in site_lon]

                    center_lat_r, center_lon_r = r["center"]
                    center_lon_r = float(center_lon_r)
                    center_lat_r = float(center_lat_r)

                    phi_r = float(phi_values[region_name])
                    theta_r = float(theta_values[region_name])

                    if phi_r > 0:
                        arrow_length = 0.32 * (phi_r ** 0.6)
                    else:
                        arrow_length = 0.0

                    u = arrow_length * np.sin(theta_r)
                    v = arrow_length * np.cos(theta_r)

                    lon0, lon1 = r["lon_range"]
                    lat0, lat1 = r["lat_range"]

                    region_rows.append({
                        "window_index": win_idx,
                        "window_id": window_id,
                        "window_end": window_end.strftime("%Y-%m-%d"),
                        "region_name": region_name,
                        "i": i,
                        "j": j,
                        "lon0": lon0,
                        "lon1": lon1,
                        "lat0": lat0,
                        "lat1": lat1,
                        "center_lon": center_lon_r,
                        "center_lat": center_lat_r,
                        "phi_region": phi_r,
                        "theta_region_rad": theta_r,
                        "theta_region_deg": np.degrees(theta_r),
                        "u": u,
                        "v": v,
                        "n_sites_in_region": len(sites_in_region),
                        "has_data": int(len(sites_in_region) > 0),
                    })

            print(f"[{win_idx}/{total_windows-1}] Processed window: {window_id}")

        except Exception as e:
            print(f"[{win_idx}/{total_windows-1}] Skipping window {window_end.strftime('%Y-%m-%d')}, reason: {e}")

        gc.collect()

    # Save CSV files
    summary_df = pd.DataFrame(summary_rows)
    site_df = pd.DataFrame(site_rows)
    region_df = pd.DataFrame(region_rows)

    summary_csv = os.path.join(output_dir, "window_summary.csv")
    site_csv = os.path.join(output_dir, "site_vectors.csv")
    region_csv = os.path.join(output_dir, "region_vectors.csv")

    summary_df.to_csv(summary_csv, index=False, encoding="utf-8-sig")
    site_df.to_csv(site_csv, index=False, encoding="utf-8-sig")
    region_df.to_csv(region_csv, index=False, encoding="utf-8-sig")

    print("\nSaving completed:")
    print(summary_csv)
    print(site_csv)
    print(region_csv)


# ========================
# 7) Entry point
# ========================
if __name__ == "__main__":
    csv_dir = r"D:\a\master\Earthquake-US\Data\20190706-102\PosData-7"
    output_dir = r"D:\a\master\Earthquake-US\Fig-Use-2\Fig-3\plot_csv_data"

    df_merged = load_and_merge_data(csv_dir)

    START_WIN = 0
    END_WIN = 702

    save_plot_data_to_csv(
        df_all=df_merged,
        output_dir=output_dir,
        window_size=30,
        step=1,
        time_step_in_window=1,
        start_win=START_WIN,
        end_win=END_WIN,
        center_lat=35.7695,
        center_lon=242.4006667,
        delta=1.0
    )
'''

In [ ]:
# (b)-(g) Use forward and reverse plots
'''
import os
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from matplotlib.ticker import AutoMinorLocator

# ========================
# 1) Read saved plotting data
# ========================
def load_plot_csv_data(csv_data_dir):
    summary_csv = os.path.join(csv_data_dir, "window_summary.csv")
    site_csv = os.path.join(csv_data_dir, "site_vectors.csv")
    region_csv = os.path.join(csv_data_dir, "region_vectors.csv")

    if not os.path.exists(summary_csv):
        raise FileNotFoundError(summary_csv)
    if not os.path.exists(site_csv):
        raise FileNotFoundError(site_csv)
    if not os.path.exists(region_csv):
        raise FileNotFoundError(region_csv)

    summary_df = pd.read_csv(summary_csv)
    site_df = pd.read_csv(site_csv)
    region_df = pd.read_csv(region_csv)

    summary_df["window_end"] = pd.to_datetime(summary_df["window_end"])
    summary_df["window_start"] = pd.to_datetime(summary_df["window_start"])
    site_df["window_end"] = pd.to_datetime(site_df["window_end"])
    region_df["window_end"] = pd.to_datetime(region_df["window_end"])

    # Convert longitude uniformly to -180 ~ 180 to ensure consistency with the background image
    if "lon" in site_df.columns:
        site_df["lon"] = np.where(site_df["lon"] > 180, site_df["lon"] - 360, site_df["lon"])

    for col in ["lon0", "lon1", "center_lon"]:
        if col in region_df.columns:
            region_df[col] = np.where(region_df[col] > 180, region_df[col] - 360, region_df[col])

    return summary_df, site_df, region_df


# ========================
# 2) Plot a single map
# ========================
def plot_single_map_from_csv(summary_row, site_sub, region_sub, bg_full_img=None, output_path=None, draw_grid=True,
                             dpi=600, figsize=(3.54, 3.95), title_fontsize=11.0):
    plt.rcParams["font.family"] = "Arial"
    plt.rcParams["mathtext.fontset"] = "stix"
    plt.rcParams["pdf.fonttype"] = 42
    plt.rcParams["ps.fonttype"] = 42

    x_label_fs = 11.0          # Increase the x-axis label font size
    y_label_fs = 11.0          # Keep the original y-axis label size; modify together if needed
    tick_fs = 11               # Keep tick label size unchanged
    legend_fs = 8.0            # Increase legend text size
    region_text_fs = 6.0

    grid_color = "#7C8CA0"
    region_text_color = "#334155"
    site_arrow_color = "#C35A4A"
    region_arrow_color = "#2A9D8F"
    station_edge_color = "#2B2B2B"

    x0 = region_sub["lon0"].min()
    x1 = region_sub["lon1"].max()
    y0 = region_sub["lat0"].min()
    y1 = region_sub["lat1"].max()

    fig, ax = plt.subplots(figsize=figsize)

    site_quiver_kw = dict(
        color=site_arrow_color,
        angles="xy",
        scale_units="xy",
        scale=0.25,
        width=0.0050,
        headwidth=3.6,
        headlength=4.2,
        headaxislength=3.7,
        alpha=0.82,
        zorder=6
    )

    region_quiver_kw = dict(
        color=region_arrow_color,
        angles="xy",
        scale_units="xy",
        scale=1.0,
        width=0.0070,
        headwidth=5.0,
        headlength=6.0,
        headaxislength=5.0,
        alpha=0.92,
        zorder=7
    )

    # Background image
    if bg_full_img is not None:
        ax.imshow(
            bg_full_img,
            extent=[x0, x1, y0, y1],
            aspect="auto",
            zorder=0,
            alpha=0.96
        )

    # 3x3 grid lines
    if draw_grid:
        for _, r in region_sub.iterrows():
            ax.add_patch(Rectangle(
                (r["lon0"], r["lat0"]),
                r["lon1"] - r["lon0"],
                r["lat1"] - r["lat0"],
                fill=False,
                linewidth=0.75,
                linestyle=(0, (3, 5)),
                edgecolor=grid_color,
                alpha=0.80,
                zorder=2
            ))

    # Station circles
    if len(site_sub) > 0:
        station_xy = site_sub[["site_id", "lon", "lat"]].drop_duplicates()
        ax.scatter(
            station_xy["lon"], station_xy["lat"],
            s=11,                           # ← Modify station circle size here
            facecolors="none",
            edgecolors=station_edge_color,
            linewidth=0.60,                # ← Modify station circle line width here
            zorder=5
        )

    # Site arrows
    for _, row in site_sub.iterrows():
        ax.quiver(row["lon"], row["lat"], -row["u"], -row["v"], **site_quiver_kw)

    # ========================
    # Modify text positions for each region here
    # First value: horizontal position (larger means farther right)
    # Second value: vertical position (larger means farther up)
    # ========================
    custom_text_pos = {
        "region_0_0": (0.03, 0.03),  # ← Modify text position for this region
        "region_0_1": (0.02, 0.58),  # ← Modify text position for this region
        "region_0_2": (0.38, 0.03),  # ← Modify text position for this region

        "region_1_0": (0.03, 0.97),  # ← Modify text position for this region
        "region_1_1": (0.03, 0.97),  # ← Modify text position for this region
        "region_1_2": (0.38, 0.97),  # ← Modify text position for this region

        "region_2_0": (0.03, 0.03),  # ← Modify text position for this region
        "region_2_1": (0.38, 0.97),  # ← Modify text position for this region
        "region_2_2": (0.03, 0.03),  # ← Modify text position for this region
    }

    # Regional arrows + regional text
    for _, row in region_sub.iterrows():
        region_name = row["region_name"]
        phi_r = float(row["phi_region"])
        theta_r = float(row["theta_region_rad"])
        theta_r_plot = ((theta_r + np.pi) % (2 * np.pi))
        theta_r_plot = np.where(theta_r_plot > np.pi, theta_r_plot - 2 * np.pi, theta_r_plot)
        has_data = int(row["has_data"])

        if has_data and phi_r > 0:
            ax.quiver(row["center_lon"], row["center_lat"], -row["u"], -row["v"], **region_quiver_kw)

        fx, fy = custom_text_pos.get(region_name, (0.03, 0.03))
        tx = row["lon0"] + fx * (row["lon1"] - row["lon0"])
        ty = row["lat0"] + fy * (row["lat1"] - row["lat0"])

        ha = "left"
        va = "top" if fy > 0.8 else "bottom"

        if not has_data:
            txt = "No Data"
            tcolor = region_text_color
        else:
            txt = (
                rf"$\phi^{{region}}$: {phi_r:.3f}" "\n"
                rf"$\theta^{{region}}$: {np.degrees(theta_r_plot):.1f}°"
            )
            tcolor = region_text_color

        ax.text(
            tx, ty, txt,
            ha=ha, va=va,
            fontsize=region_text_fs,      # ← Modify regional text font size here
            color=tcolor,                 # ← Modify regional text color here
            weight="normal",
            multialignment="left",
            linespacing=1.00,
            bbox=dict(
                boxstyle="round,pad=0.08",
                facecolor=(1, 1, 1, 0.28),
                edgecolor="none"
            ),
            zorder=8
        )

    ax.set_xlim(x0, x1)
    ax.set_ylim(y0, y1)
    ax.set_xticks([-118.8, -118.0, -117.2, -116.4])
    ax.set_xlabel("Longitude (°E)", fontsize=x_label_fs)
    ax.set_ylabel("Latitude (°N)", fontsize=y_label_fs)
    ax.xaxis.set_minor_locator(AutoMinorLocator(5))
    ax.yaxis.set_minor_locator(AutoMinorLocator(5))
    
    ax.tick_params(axis="both", which="major", labelsize=tick_fs, width=0.6, length=3)
    ax.tick_params(axis="both", which="minor", width=0.5, length=1.8)
    for spine in ax.spines.values():
        spine.set_linewidth(0.6)
        spine.set_color("black")

    center_str = pd.to_datetime(summary_row["window_end"]).strftime("%Y-%m-%d")
    phi_global = float(summary_row["phi_global"])
    top_text = rf"{center_str}    $\phi^{{global}}$ = {phi_global:.3f}"

    fig.suptitle(
        top_text,
        fontsize=title_fontsize,
        y=0.92                     # ← Modify title distance from the image here; smaller values move it closer
    )

    legend_elements = [
        Line2D([0], [0], marker="o", color="blue", label="Stations",
               markerfacecolor="none", markeredgecolor=station_edge_color, markersize=4.6, linewidth=0),
        Line2D([0], [0], marker=r"$\rightarrow$", color=site_arrow_color, label="Site",
               markerfacecolor=site_arrow_color, markersize=8.3, linewidth=0),
        Line2D([0], [0], marker=r"$\rightarrow$", color=region_arrow_color, label="Regional",
               markerfacecolor=region_arrow_color, markersize=8.3, linewidth=0),
    ]
    ax.legend(
        handles=legend_elements,
        loc="upper right",           # ← Modify the overall legend position here
        fontsize=legend_fs,
        frameon=False
    )

    plt.tight_layout(rect=[0, 0, 1, 0.985])   # ← Also adjust the overall spacing between the title and figure here

    if output_path:
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        fig.savefig(output_path, dpi=dpi, bbox_inches="tight")

    plt.close(fig)
    gc.collect()


# ========================
# 3) Batch plotting
# ========================
def plot_all_maps_from_csv(csv_data_dir, output_dir, bg_full_path=None, start_win=0, end_win=None, dpi=300):
    summary_df, site_df, region_df = load_plot_csv_data(csv_data_dir)

    os.makedirs(output_dir, exist_ok=True)

    png_dir = os.path.join(output_dir, "single_maps_from_csv_png")
    pdf_dir = os.path.join(output_dir, "single_maps_from_csv_pdf")

    os.makedirs(png_dir, exist_ok=True)
    os.makedirs(pdf_dir, exist_ok=True)

    if bg_full_path is not None and os.path.exists(bg_full_path):
        bg_full_img = Image.open(bg_full_path).convert("RGB")
    else:
        bg_full_img = None
        print("Warning: The background image does not exist and will not be loaded.")

    summary_df = summary_df.sort_values("window_index").reset_index(drop=True)
    total_windows = len(summary_df)

    if total_windows == 0:
        raise ValueError("No plottable data found in window_summary.csv")

    if start_win is None:
        start_win = 0
    start_win = max(0, int(start_win))

    if end_win is None:
        end_win = total_windows - 1
    end_win = min(total_windows - 1, int(end_win))

    if start_win > end_win:
        raise ValueError(f"start_win({start_win}) > end_win({end_win})")

    print(f"Drawing windows: {start_win} -> {end_win}")

    for idx in range(start_win, end_win + 1):
        summary_row = summary_df.iloc[idx]
        win_idx = int(summary_row["window_index"])
        window_id = summary_row["window_id"]

        site_sub = site_df[site_df["window_index"] == win_idx].copy()
        region_sub = region_df[region_df["window_index"] == win_idx].copy()

        out_png = os.path.join(png_dir, f"single_{window_id}.png")
        out_pdf = os.path.join(pdf_dir, f"single_{window_id}.pdf")

        # Save PNG
        plot_single_map_from_csv(
            summary_row=summary_row,
            site_sub=site_sub,
            region_sub=region_sub,
            bg_full_img=bg_full_img,
            output_path=out_png,
            draw_grid=True,
            dpi=dpi
        )

        # Save PDF
        plot_single_map_from_csv(
            summary_row=summary_row,
            site_sub=site_sub,
            region_sub=region_sub,
            bg_full_img=bg_full_img,
            output_path=out_pdf,
            draw_grid=True,
            dpi=dpi
        )

        print(f"[{idx}/{end_win}] Saved PNG: {out_png}")
        print(f"[{idx}/{end_win}] Saved PDF: {out_pdf}")

    if bg_full_img is not None:
        bg_full_img.close()


# ========================
# 4) Entry point
# ========================
if __name__ == "__main__":
    csv_data_dir = r"D:\a\master\Earthquake-US\Fig-Use-2\Fig-3\plot_csv_data"
    output_dir = r"D:\a\master\Earthquake-US\Fig-Use-2\Fig-3\180"
    bg_full_path = r"D:\a\master\Earthquake-US\Fig-Use-2\Fig-3\Background_PyGMT_only_topo_fault.png"

    START_WIN = 415
    END_WIN = 433

    plot_all_maps_from_csv( csv_data_dir=csv_data_dir, output_dir=output_dir, bg_full_path=bg_full_path,
                            start_win=START_WIN, end_win=END_WIN, dpi=300)
'''

Fig-4

In [ ]:
#The data of V
'''
# -*- coding: utf-8 -*-

import numpy as np
import pandas as pd
import glob
import os
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import AutoMinorLocator, MultipleLocator
from matplotlib.patches import Rectangle


# =========================================================
# 0. Plotting style parameters
# =========================================================
FIG_WIDTH = 7.2
FIG_HEIGHT = 2.5

plt.rcParams["font.family"] = "Arial"
plt.rcParams["mathtext.fontset"] = "stix"
plt.rcParams["font.size"] = 12
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["xtick.labelsize"] = 11
plt.rcParams["ytick.labelsize"] = 11
plt.rcParams["legend.fontsize"] = 11

plt.rcParams["axes.linewidth"] = 0.7
plt.rcParams["xtick.major.width"] = 0.7
plt.rcParams["ytick.major.width"] = 0.7
plt.rcParams["xtick.minor.width"] = 0.5
plt.rcParams["ytick.minor.width"] = 0.5

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["savefig.facecolor"] = "white"
plt.rcParams["savefig.transparent"] = False


# =========================================================
# 1. Common functions
# =========================================================
def linear_fit_slope(y, x=None):
    """
    Fit a one-dimensional sequence:

        y = a*x + b

    Return the slope a.
    """
    y = np.asarray(y, dtype=float)

    if x is None:
        x = np.arange(len(y), dtype=float)
    else:
        x = np.asarray(x, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)

    if mask.sum() < 2:
        return np.nan

    if np.nanstd(y[mask]) == 0:
        return 0.0

    coef = np.polyfit(x[mask], y[mask], 1)

    a = coef[0]

    return a


def linear_fit_slope_and_intercept(y, x=None):
    """
    Fit a one-dimensional sequence:

        y = a*x + b

    Return:
        a, b
    """
    y = np.asarray(y, dtype=float)

    if x is None:
        x = np.arange(len(y), dtype=float)
    else:
        x = np.asarray(x, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)

    if mask.sum() < 2:
        return np.nan, np.nan

    if np.nanstd(y[mask]) == 0:
        return 0.0, np.nanmean(y[mask])

    coef = np.polyfit(x[mask], y[mask], 1)

    a = coef[0]
    b = coef[1]

    return a, b


def linear_fit_slopes_2d_by_column(A):
    """
    Fit a linear trend to each column of a two-dimensional matrix.

    A shape:
        Time × number of sites

    Fit each column:
        y_j = a_j*x + b_j

    Return:
        slope a_j for each column
    """
    A = np.asarray(A, dtype=float)

    slopes = []

    for j in range(A.shape[1]):
        slope_j = linear_fit_slope(A[:, j])
        slopes.append(slope_j)

    return np.asarray(slopes, dtype=float)


def standardize_1d(y):
    """
    Standardization:
        y_standardized = (y - mean) / std
    """
    y = np.asarray(y, dtype=float)

    mean_val = np.nanmean(y)
    std_val = np.nanstd(y)

    if not np.isfinite(std_val) or std_val == 0:
        return np.zeros_like(y)

    return (y - mean_val) / std_val


def center_1d(y):
    """
    Mean centering:
        y_centered = y - mean
    """
    y = np.asarray(y, dtype=float)

    mean_val = np.nanmean(y)

    if not np.isfinite(mean_val):
        return np.zeros_like(y)

    return y - mean_val


def load_and_merge_data(csv_dir):
    """
    Load and merge all CSV files.
    Each CSV filename is used as the site_id.
    """
    files = glob.glob(f"{csv_dir}/*.csv")

    if len(files) == 0:
        raise FileNotFoundError(f"No CSV files found in folder: {csv_dir}")

    dfs = []

    for file in files:
        df = pd.read_csv(file)
        df["site_id"] = os.path.basename(file).split(".")[0]
        dfs.append(df)

    return pd.concat(dfs, ignore_index=True)


def compute_eigen_microstates(A):
    """
    Calculate eigen/eigenmicrostates using SVD.

    A shape:
        Time length M × spatial dimension N_T

    Perform SVD on A.T:
        A.T = U S Vt

    where:
        V = Vt.T

    V[:, 0] is the first principal-component time series.
    """
    U, S, Vt = np.linalg.svd(A.T, full_matrices=False)

    eigenvalues = S ** 2
    eigenmicrostates = U
    V = Vt.T

    return eigenvalues, eigenmicrostates, V, S


# =========================================================
# 2. Standardized version: used for IMS / V[:, 0]
# =========================================================
def preprocess_gnss_data_standardized(df, time_step=1):
    """
    Standardized version.

    Workflow:
        Raw dE/dN
        -> standardize each site separately
        -> construct A matrix
        -> normalize the entire A matrix
    """
    if "YYYYMMDD" not in df.columns:
        raise ValueError("DataFrame must contain the 'YYYYMMDD' column")

    df = df.copy()

    if "time" not in df.columns:
        df["time"] = pd.to_datetime(
            df["YYYYMMDD"],
            format="%Y%m%d"
        ).dt.tz_localize(None)
    else:
        df["time"] = pd.to_datetime(df["time"]).dt.tz_localize(None)

    df = df.sort_values(["site_id", "time"])

    sampled_times = np.sort(df["time"].unique())[::time_step]
    df_sampled = df[df["time"].isin(sampled_times)].copy()

    for component in ["dE", "dN"]:
        if component not in df_sampled.columns:
            raise ValueError(f"DataFrame must contain the '{component}' column")

        standardized_col = f"{component}_standardized"
        df_sampled[standardized_col] = 0.0

        for site in df_sampled["site_id"].unique():
            site_mask = df_sampled["site_id"] == site
            values = df_sampled.loc[site_mask, component].values

            df_sampled.loc[site_mask, standardized_col] = standardize_1d(values)

    site_lengths = df_sampled.groupby("site_id").size()

    if len(site_lengths.unique()) > 1:
        print("Warning: Data lengths are inconsistent among sites; automatically trimming to the shortest length")
        M = site_lengths.min()
    else:
        M = site_lengths.iloc[0]

    sites = sorted(df_sampled["site_id"].unique())
    N_T = 2 * len(sites)

    A = np.zeros((M, N_T))

    for i, site in enumerate(sites):
        site_data = (
            df_sampled[df_sampled["site_id"] == site]
            .sort_values("time")
            .iloc[:M]
        )

        A[:, 2 * i] = site_data["dE_standardized"].values
        A[:, 2 * i + 1] = site_data["dN_standardized"].values

    C_0 = np.sum(A ** 2)

    if C_0 > 0:
        A_normalized = A / np.sqrt(C_0)
    else:
        A_normalized = A

    return A_normalized, sites, sampled_times[:M], df_sampled


def generate_ims_slope_time_series(df, window_size=30, step=1):
    """
    Calculate the IMS slope time series.

    Workflow:
        Each 30-day window
        -> standardized preprocessing
        -> construct A matrix
        -> SVD
        -> take V[:, 0]
        -> fit V[:, 0] = a*x + b
        -> save a

    Note:
        The SVD result V[:, 0] may be multiplied entirely by -1.
        To keep the sign of the slope continuous, V[:, 0] is aligned
        with that from the previous window.
    """
    df = df.copy()

    if "time" not in df.columns:
        df["time"] = pd.to_datetime(
            df["YYYYMMDD"],
            format="%Y%m%d"
        ).dt.tz_localize(None)
    else:
        df["time"] = pd.to_datetime(df["time"]).dt.tz_localize(None)

    all_times = np.sort(pd.to_datetime(df["time"].unique()))

    window_end_dates = []
    slope_ims_values = []
    intercept_ims_values = []

    prev_v_first_component = None

    for start in range(0, len(all_times) - window_size + 1, step):
        window_times = all_times[start:start + window_size]
        window_end = window_times[-1]

        window_end_dates.append(window_end)

        df_window = df[df["time"].isin(window_times)].copy()

        A, sites, _, df_sampled = preprocess_gnss_data_standardized(df_window)

        if A.shape[0] == 0 or A.shape[1] == 0:
            slope_ims_values.append(np.nan)
            intercept_ims_values.append(np.nan)
            continue

        eigenvalues, eigenmicrostates, V, S = compute_eigen_microstates(A)

        if V is None or V.shape[0] < 2:
            slope_ims_values.append(np.nan)
            intercept_ims_values.append(np.nan)
            continue

        v_first_component = V[:, 0].astype(float)

        # =================================================
        # Handle SVD sign ambiguity
        # If V[:, 0] in the current window is opposite to that in the previous window, multiply it by -1
        # =================================================
        if prev_v_first_component is not None:
            n = min(len(v_first_component), len(prev_v_first_component))

            dot_val = np.dot(
                v_first_component[:n],
                prev_v_first_component[:n]
            )

            if dot_val < 0:
                v_first_component = -v_first_component
        else:
            # Use a fixed sign convention for the first window
            if np.nanmean(v_first_component) < 0:
                v_first_component = -v_first_component

        prev_v_first_component = v_first_component.copy()

        # Fit V[:, 0] = a*x + b
        slope_ims, intercept_ims = linear_fit_slope_and_intercept(v_first_component)

        slope_ims_values.append(slope_ims)
        intercept_ims_values.append(intercept_ims)

    return window_end_dates, slope_ims_values, intercept_ims_values


# =========================================================
# 3. Mean-centered version: used for dE / dN
# =========================================================
def preprocess_gnss_data_centered(df, time_step=1):
    """
    Mean-centered version.

    Workflow:
        Raw dE/dN
        -> mean-center each site separately
        -> construct A matrix
        -> normalize the entire A matrix
    """
    if "YYYYMMDD" not in df.columns:
        raise ValueError("DataFrame must contain the 'YYYYMMDD' column")

    df = df.copy()

    if "time" not in df.columns:
        df["time"] = pd.to_datetime(
            df["YYYYMMDD"],
            format="%Y%m%d"
        ).dt.tz_localize(None)
    else:
        df["time"] = pd.to_datetime(df["time"]).dt.tz_localize(None)

    df = df.sort_values(["site_id", "time"])

    sampled_times = np.sort(df["time"].unique())[::time_step]
    df_sampled = df[df["time"].isin(sampled_times)].copy()

    for component in ["dE", "dN"]:
        if component not in df_sampled.columns:
            raise ValueError(f"DataFrame must contain the '{component}' column")

        centered_col = f"{component}_centered"
        df_sampled[centered_col] = 0.0

        for site in df_sampled["site_id"].unique():
            site_mask = df_sampled["site_id"] == site
            values = df_sampled.loc[site_mask, component].values

            df_sampled.loc[site_mask, centered_col] = center_1d(values)

    site_lengths = df_sampled.groupby("site_id").size()

    if len(site_lengths.unique()) > 1:
        print("Warning: Data lengths are inconsistent for some sites; automatically trimming to the shortest length")
        M = site_lengths.min()
    else:
        M = site_lengths.iloc[0]

    sites = sorted(df_sampled["site_id"].unique())
    N_T = 2 * len(sites)

    A = np.zeros((M, N_T))

    for i, site in enumerate(sites):
        site_data = (
            df_sampled[df_sampled["site_id"] == site]
            .sort_values("time")
            .iloc[:M]
        )

        A[:, 2 * i] = site_data["dE_centered"].values
        A[:, 2 * i + 1] = site_data["dN_centered"].values

    C_0 = np.sum(A ** 2)

    if C_0 > 0:
        A_normalized = A / np.sqrt(C_0)
    else:
        A_normalized = A

    return A_normalized, sites, sampled_times[:M], df_sampled


def generate_de_dn_slope_time_series(df, window_size=30, step=1):
    """
    Calculate the dE / dN slope time series.

    Workflow:
        Each 30-day window
        -> mean-centered preprocessing
        -> construct A matrix
        -> separate A_dE and A_dN
        -> fit a*x+b to each column of A_dE
        -> fit a*x+b to each column of A_dN
        -> take the mean slope across all sites separately
    """
    df = df.copy()

    if "time" not in df.columns:
        df["time"] = pd.to_datetime(
            df["YYYYMMDD"],
            format="%Y%m%d"
        ).dt.tz_localize(None)
    else:
        df["time"] = pd.to_datetime(df["time"]).dt.tz_localize(None)

    all_times = np.sort(pd.to_datetime(df["time"].unique()))

    window_end_dates = []

    slope_dE_mean_values = []
    slope_dN_mean_values = []

    slope_dE_median_values = []
    slope_dN_median_values = []

    slope_dE_abs_mean_values = []
    slope_dN_abs_mean_values = []

    for start in range(0, len(all_times) - window_size + 1, step):
        window_times = all_times[start:start + window_size]
        window_end = window_times[-1]

        window_end_dates.append(window_end)

        df_window = df[df["time"].isin(window_times)].copy()

        A, sites, _, df_sampled = preprocess_gnss_data_centered(df_window)

        if A.shape[0] == 0 or A.shape[1] == 0:
            slope_dE_mean_values.append(np.nan)
            slope_dN_mean_values.append(np.nan)
            slope_dE_median_values.append(np.nan)
            slope_dN_median_values.append(np.nan)
            slope_dE_abs_mean_values.append(np.nan)
            slope_dN_abs_mean_values.append(np.nan)
            continue

        # Even-numbered columns of A are dE, odd-numbered columns are dN
        A_dE = A[:, 0::2]
        A_dN = A[:, 1::2]

        # Fit y = a*x + b for each site separately and extract a
        slopes_dE = linear_fit_slopes_2d_by_column(A_dE)
        slopes_dN = linear_fit_slopes_2d_by_column(A_dN)

        # To obtain one dE curve and one dN curve, combine the slopes from multiple sites into one value
        # The main curves use the mean here
        slope_dE_mean = np.nanmean(slopes_dE)
        slope_dN_mean = np.nanmean(slopes_dN)

        slope_dE_median = np.nanmedian(slopes_dE)
        slope_dN_median = np.nanmedian(slopes_dN)

        slope_dE_abs_mean = np.nanmean(np.abs(slopes_dE))
        slope_dN_abs_mean = np.nanmean(np.abs(slopes_dN))

        slope_dE_mean_values.append(slope_dE_mean)
        slope_dN_mean_values.append(slope_dN_mean)

        slope_dE_median_values.append(slope_dE_median)
        slope_dN_median_values.append(slope_dN_median)

        slope_dE_abs_mean_values.append(slope_dE_abs_mean)
        slope_dN_abs_mean_values.append(slope_dN_abs_mean)

    return (
        window_end_dates,
        slope_dE_mean_values,
        slope_dN_mean_values,
        slope_dE_median_values,
        slope_dN_median_values,
        slope_dE_abs_mean_values,
        slope_dN_abs_mean_values,
    )


# =========================================================
# 4. Combined analysis
# =========================================================
def analyze_three_slope_series(df):
    """
    Calculate three slope curves simultaneously:
        1. IMS slope
        2. dE slope
        3. dN slope
    """
    print("Starting calculation: IMS slope, workflow: A matrix -> SVD -> V[:,0] -> fit ax+b -> a ...")

    dates_ims, slope_ims, intercept_ims = generate_ims_slope_time_series(
        df,
        window_size=30,
        step=1
    )

    print("Starting calculation: dE / dN slope, workflow: A matrix -> A_dE/A_dN -> fit ax+b for each column -> mean(a) ...")

    (
        dates_initial,
        slope_dE_mean,
        slope_dN_mean,
        slope_dE_median,
        slope_dN_median,
        slope_dE_abs_mean,
        slope_dN_abs_mean,
    ) = generate_de_dn_slope_time_series(
        df,
        window_size=30,
        step=1
    )

    return (
        dates_ims,
        slope_ims,
        intercept_ims,
        dates_initial,
        slope_dE_mean,
        slope_dN_mean,
        slope_dE_median,
        slope_dN_median,
        slope_dE_abs_mean,
        slope_dN_abs_mean,
    )


# =========================================================
# 5. Save slope data
# =========================================================
def save_slope_data_to_csv(
    dates_ims,
    slope_ims,
    intercept_ims,
    dates_initial,
    slope_dE_mean,
    slope_dN_mean,
    slope_dE_median,
    slope_dN_median,
    slope_dE_abs_mean,
    slope_dN_abs_mean,
    output_csv
):
    max_len = max(
        len(dates_ims),
        len(slope_ims),
        len(intercept_ims),
        len(dates_initial),
        len(slope_dE_mean),
        len(slope_dN_mean),
        len(slope_dE_median),
        len(slope_dN_median),
        len(slope_dE_abs_mean),
        len(slope_dN_abs_mean)
    )

    def pad_list(x, length):
        x = list(x)
        return x + [np.nan] * (length - len(x))

    out_df = pd.DataFrame({
        "date_ims": pad_list(pd.to_datetime(dates_ims).strftime("%Y-%m-%d"), max_len),
        "slope_ims": pad_list(slope_ims, max_len),
        "intercept_ims": pad_list(intercept_ims, max_len),

        "date_initial": pad_list(pd.to_datetime(dates_initial).strftime("%Y-%m-%d"), max_len),
        "slope_dE": pad_list(slope_dE_mean, max_len),
        "slope_dN": pad_list(slope_dN_mean, max_len),

        # The following are backup columns for checking
        "slope_dE_median": pad_list(slope_dE_median, max_len),
        "slope_dN_median": pad_list(slope_dN_median, max_len),
        "slope_dE_abs_mean": pad_list(slope_dE_abs_mean, max_len),
        "slope_dN_abs_mean": pad_list(slope_dN_abs_mean, max_len),
    })

    out_df.to_csv(output_csv, index=False, encoding="utf-8-sig")

    print(f"Slope data saved to: {output_csv}")


# =========================================================
# 6. Plotting functions
# =========================================================
def style_time_series_axis(ax, ylabel, y_major=None, y_minor_n=5, ylim=None):
    ax.set_xlabel("Date", labelpad=2)
    ax.set_ylabel(ylabel, labelpad=2)

    major_ticks = pd.to_datetime([
        "2018-12-01",
        "2019-08-01",
        "2020-04-01"
    ])

    ax.set_xticks(major_ticks)
    ax.xaxis.set_minor_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

    if y_major is not None:
        ax.yaxis.set_major_locator(MultipleLocator(y_major))

    ax.yaxis.set_minor_locator(AutoMinorLocator(y_minor_n))

    if ylim is not None:
        ax.set_ylim(*ylim)

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.7)
        spine.set_color("black")

    ax.tick_params(
        axis="both",
        which="major",
        direction="in",
        length=3.5,
        width=0.7,
        pad=2,
        top=False,
        right=False,
        bottom=True,
        left=True
    )

    ax.tick_params(
        axis="both",
        which="minor",
        direction="in",
        length=2.0,
        width=0.5,
        top=False,
        right=False,
        bottom=True,
        left=True
    )

    ax.grid(False)
    ax.margins(x=0.01)


def style_right_axis(ax2, ylabel, y_major=None, y_minor_n=5, ylim=None):
    ax2.set_ylabel(ylabel, labelpad=2)

    if y_major is not None:
        ax2.yaxis.set_major_locator(MultipleLocator(y_major))

    ax2.yaxis.set_minor_locator(AutoMinorLocator(y_minor_n))

    if ylim is not None:
        ax2.set_ylim(*ylim)

    for spine in ax2.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.7)
        spine.set_color("black")

    ax2.tick_params(
        axis="y",
        which="major",
        direction="in",
        length=3.5,
        width=0.7,
        pad=2,
        right=True,
        left=False
    )

    ax2.tick_params(
        axis="y",
        which="minor",
        direction="in",
        length=2.0,
        width=0.5,
        right=True,
        left=False
    )

    # Prevent duplicate x-axis ticks on the right axis
    ax2.tick_params(
        axis="x",
        which="both",
        bottom=False,
        top=False,
        labelbottom=False
    )


def add_four_boxes(ax, y_bottom, y_top, blue_y_bottom=None, blue_y_top=None):
    boxes = [
        ("2019-05-03", "2019-06-30", "#ffbe0b", y_bottom, y_top),
        ("2019-07-03", "2019-08-05", "#fb5607", y_bottom, y_top),
        ("2019-08-09", "2019-10-31", "#8338ec", y_bottom, y_top),
        (
            "2019-05-25",
            "2019-06-18",
            "#3a86ff",
            blue_y_bottom if blue_y_bottom is not None else y_bottom,
            blue_y_top if blue_y_top is not None else y_top
        )
    ]

    for start_date, end_date, color, y0, y1 in boxes:
        x0 = mdates.date2num(pd.to_datetime(start_date))
        x1 = mdates.date2num(pd.to_datetime(end_date))

        rect = Rectangle(
            (x0, y0),
            x1 - x0,
            y1 - y0,
            fill=False,
            edgecolor=color,
            linewidth=1.3,
            linestyle=(0, (7, 3.5)),
            zorder=1.5
        )

        ax.add_patch(rect)


def plot_slope(slope_csv, output_dir):
    df = pd.read_csv(slope_csv)

    dates_ims = pd.to_datetime(df["date_ims"], errors="coerce")
    dates_initial = pd.to_datetime(df["date_initial"], errors="coerce")

    slope_ims = pd.to_numeric(df["slope_ims"], errors="coerce")
    slope_dE = pd.to_numeric(df["slope_dE"], errors="coerce")
    slope_dN = pd.to_numeric(df["slope_dN"], errors="coerce")

    fig, ax = plt.subplots(
        1,
        1,
        figsize=(FIG_WIDTH, FIG_HEIGHT),
        constrained_layout=False
    )

    # =====================================================
    # Left axis: IMS slope
    # =====================================================
    line_ims, = ax.plot(
        dates_ims,
        slope_ims,
        color="#1b5e9a",
        linewidth=1.35,
        solid_capstyle="round",
        solid_joinstyle="round",
        label="IMS slope",
        zorder=3
    )

    style_time_series_axis(
        ax=ax,
        ylabel="IMS slope",
        y_major=None,
        y_minor_n=5,
        ylim=None
    )

    ax.axhline(
        0,
        color="#1b5e9a",
        linewidth=0.6,
        linestyle="--",
        alpha=0.75,
        zorder=1
    )

    # =====================================================
    # Right axis: dE slope and dN slope
    # =====================================================
    ax2 = ax.twinx()

    line_dE, = ax2.plot(
        dates_initial,
        slope_dE,
        color="#b7652b",
        linewidth=1.05,
        alpha=0.95,
        solid_capstyle="round",
        solid_joinstyle="round",
        label="dE slope",
        zorder=2
    )

    line_dN, = ax2.plot(
        dates_initial,
        slope_dN,
        color="#4f8a5b",
        linewidth=1.05,
        alpha=0.95,
        solid_capstyle="round",
        solid_joinstyle="round",
        label="dN slope",
        zorder=2
    )

    style_right_axis(
        ax2=ax2,
        ylabel="dE / dN slope",
        y_major=None,
        y_minor_n=5,
        ylim=None
    )

    ax2.axhline(
        0,
        color="gray",
        linewidth=0.6,
        linestyle="--",
        alpha=0.65,
        zorder=1
    )

    # Make the left-axis background transparent to avoid covering the right-axis curves
    ax.patch.set_visible(False)

    # =====================================================
    # Combine the legend and place it in the upper-right corner
    # =====================================================
    ax.legend(
        handles=[line_ims, line_dE, line_dN],
        loc="upper right",
        frameon=False,
        handlelength=1.4,
        handletextpad=0.5,
        borderpad=0.2,
        labelspacing=0.35
    )

    # =====================================================
    # Four time boxes: drawn according to the y range of the left IMS axis
    # =====================================================
    y0, y1 = ax.get_ylim()
    yrange = y1 - y0

    box_y_bottom = y0 + 0.05 * yrange
    box_y_top = y1 - 0.05 * yrange

    blue_y_bottom = y0 + 0.15 * yrange
    blue_y_top = y0 + 0.60 * yrange

    add_four_boxes(
        ax=ax,
        y_bottom=box_y_bottom,
        y_top=box_y_top,
        blue_y_bottom=blue_y_bottom,
        blue_y_top=blue_y_top
    )

    plt.subplots_adjust(
        left=0.09,
        right=0.91,
        bottom=0.20,
        top=0.98
    )

    png_path = os.path.join(output_dir, "Slope_three_curves_double_y.png")
    pdf_path = os.path.join(output_dir, "Slope_three_curves_double_y.pdf")

    fig.savefig(png_path, dpi=600, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")

    plt.close(fig)

    print(f"Saved: {png_path}")
    print(f"Saved: {pdf_path}")


# =========================================================
# 7. Main program
# =========================================================
if __name__ == "__main__":

    output_dir = r"D:\a\master\Earthquake-US\Fig-Use-2\Fig-new\Slope(QQS)"
    csv_dir = r"D://a//master//Earthquake-US//Data//20190706-102//PosData-7"

    os.makedirs(output_dir, exist_ok=True)

    try:
        print("Loading data...")

        df_merged = load_and_merge_data(csv_dir)

        print(f"Loaded {len(df_merged)} records")
        print(f"Number of sites: {df_merged['site_id'].nunique()}")

        (
            dates_ims,
            slope_ims,
            intercept_ims,
            dates_initial,
            slope_dE_mean,
            slope_dN_mean,
            slope_dE_median,
            slope_dN_median,
            slope_dE_abs_mean,
            slope_dN_abs_mean,
        ) = analyze_three_slope_series(df_merged)

        output_csv = os.path.join(
            output_dir,
            "Slope_three_curves_plot_data_after_A_or_V.csv"
        )

        save_slope_data_to_csv(
            dates_ims,
            slope_ims,
            intercept_ims,
            dates_initial,
            slope_dE_mean,
            slope_dN_mean,
            slope_dE_median,
            slope_dN_median,
            slope_dE_abs_mean,
            slope_dN_abs_mean,
            output_csv
        )

        plot_slope(
            slope_csv=output_csv,
            output_dir=output_dir
        )

        print("Slope data calculation and plotting completed!")

    except Exception as e:
        print(f"Error: {str(e)}")

        import traceback
        traceback.print_exc()
'''

In [ ]:
# (a) Time-series trend plot, absolute values + smoothing
'''
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from matplotlib.ticker import AutoMinorLocator, FuncFormatter, MaxNLocator
from matplotlib.patches import Rectangle


# =========================================================
# 0. Plotting style parameters
# =========================================================
FIG_WIDTH = 7.2
FIG_HEIGHT = 2.5

plt.rcParams["font.family"] = "Arial"
plt.rcParams["mathtext.fontset"] = "stix"
plt.rcParams["font.size"] = 12
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["xtick.labelsize"] = 11
plt.rcParams["ytick.labelsize"] = 11

plt.rcParams["axes.linewidth"] = 0.7
plt.rcParams["xtick.major.width"] = 0.7
plt.rcParams["ytick.major.width"] = 0.7
plt.rcParams["xtick.minor.width"] = 0.5
plt.rcParams["ytick.minor.width"] = 0.5

plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["savefig.facecolor"] = "white"
plt.rcParams["savefig.transparent"] = False


# =========================================================
# 1. Smoothing function
# =========================================================
def smooth_series(y, method="savgol", window=31, polyorder=2, rolling_center=True):
    y = pd.Series(y).copy()

    if window < 3:
        return y

    if window % 2 == 0:
        window += 1

    valid_count = y.notna().sum()

    if valid_count < window:
        return y

    if method == "savgol":
        try:
            from scipy.signal import savgol_filter

            y_interp = y.interpolate(limit_direction="both")

            y_smooth = savgol_filter(
                y_interp.values,
                window_length=window,
                polyorder=min(polyorder, window - 1),
                mode="interp"
            )

            return pd.Series(y_smooth, index=y.index)

        except Exception:
            return (
                y.rolling(
                    window=window,
                    center=rolling_center,
                    min_periods=max(1, window // 3)
                )
                .mean()
                .interpolate(limit_direction="both")
            )

    elif method == "rolling":
        return (
            y.rolling(
                window=window,
                center=rolling_center,
                min_periods=max(1, window // 3)
            )
            .mean()
            .interpolate(limit_direction="both")
        )

    else:
        return y


# =========================================================
# 2. Axis tick formatting
# =========================================================
def scaled_tick_formatter(scale):
    """
    Display original values after scaling.

    For example:
    if the original value is 0.023 and scale=1e2,
    the tick label is displayed as 2.3;
    meanwhile, ×10^{-2} is uniformly indicated at the bottom of the axis.
    """
    def formatter(x, pos):
        value = x * scale

        if abs(value) < 1e-12:
            return "0"

        text = f"{value:.2f}".rstrip("0").rstrip(".")
        return text

    return FuncFormatter(formatter)


# =========================================================
# 3. Time boxes
# =========================================================
def add_four_boxes(ax, y_bottom, y_top, blue_y_bottom=None, blue_y_top=None):
    boxes = [
        ("2019-05-03", "2019-06-30", "#ffbe0b", y_bottom, y_top),
        ("2019-07-03", "2019-08-05", "#fb5607", y_bottom, y_top),
        ("2019-08-09", "2019-10-31", "#8338ec", y_bottom, y_top),
        (
            "2019-05-25",
            "2019-06-18",
            "#3a86ff",
            blue_y_bottom if blue_y_bottom is not None else y_bottom,
            blue_y_top if blue_y_top is not None else y_top
        )
    ]

    for start_date, end_date, color, y0, y1 in boxes:
        x0 = mdates.date2num(pd.to_datetime(start_date))
        x1 = mdates.date2num(pd.to_datetime(end_date))

        rect = Rectangle(
            (x0, y0),
            x1 - x0,
            y1 - y0,
            fill=False,
            edgecolor=color,
            linewidth=1.3,
            linestyle=(0, (7, 3.5)),
            zorder=1.5
        )

        ax.add_patch(rect)


# =========================================================
# 4. Plotting function: only read the existing CSV for plotting
# =========================================================
def plot_slope_from_saved_csv(
    slope_csv,
    output_dir,
    smooth=True,
    smooth_method="savgol",
    smooth_window=31,
    polyorder=2
):
    # ------------------------
    # Colors
    # ------------------------
    color_ims = "#1b5e9a"
    color_dE = "#b7652b"
    color_dN = "#4f8a5b"

    # ------------------------
    # Read existing plotting data
    # ------------------------
    df = pd.read_csv(slope_csv)

    dates_ims = pd.to_datetime(df["date_ims"], errors="coerce")
    dates_initial = pd.to_datetime(df["date_initial"], errors="coerce")

    # Use absolute slope values
    if "slope_ims_abs" in df.columns:
        slope_ims = pd.to_numeric(df["slope_ims_abs"], errors="coerce")
    else:
        slope_ims = np.abs(pd.to_numeric(df["slope_ims"], errors="coerce"))

    slope_dE = pd.to_numeric(df["slope_dE_abs_mean"], errors="coerce")
    slope_dN = pd.to_numeric(df["slope_dN_abs_mean"], errors="coerce")

    # ------------------------
    # Plot only the specified time range
    # ------------------------
    plot_start = pd.to_datetime("2018-09-06")
    plot_end = pd.to_datetime("2020-05-06")

    mask_ims = (dates_ims >= plot_start) & (dates_ims <= plot_end)
    mask_dn = (dates_initial >= plot_start) & (dates_initial <= plot_end)

    dates_ims = dates_ims[mask_ims]
    slope_ims = slope_ims[mask_ims]

    dates_initial = dates_initial[mask_dn]
    slope_dE = slope_dE[mask_dn]
    slope_dN = slope_dN[mask_dn]

    # ------------------------
    # Smoothing
    # ------------------------
    if smooth:
        slope_ims = smooth_series(
            slope_ims,
            method=smooth_method,
            window=smooth_window,
            polyorder=polyorder
        )

        slope_dE = smooth_series(
            slope_dE,
            method=smooth_method,
            window=smooth_window,
            polyorder=polyorder
        )

        slope_dN = smooth_series(
            slope_dN,
            method=smooth_method,
            window=smooth_window,
            polyorder=polyorder
        )

    # ------------------------
    # Create figure
    # ------------------------
    fig, ax = plt.subplots(
        1,
        1,
        figsize=(FIG_WIDTH, FIG_HEIGHT),
        constrained_layout=False
    )

    # =====================================================
    # Left axis: |k_IMS|
    # =====================================================
    ax.plot(
        dates_ims,
        slope_ims,
        color=color_ims,
        linewidth=1.35,
        solid_capstyle="round",
        solid_joinstyle="round",
        zorder=3
    )

    ax.set_xlabel("Date", labelpad=2)

    # Only make the left-axis label blue; ticks and values remain black
    ax.set_ylabel(
         r"$|k_{V_1}|$",
        labelpad=3,
        color=color_ims
    )

    # Display left-axis values as a, corresponding to a × 10^-2
    ax.yaxis.set_major_formatter(scaled_tick_formatter(1e2))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
    ax.yaxis.set_minor_locator(AutoMinorLocator(5))

    # Place the left-axis scale factor at the bottom
    ax.text(
        -0.075,
        0.0,
        r"$\times 10^{-2}$",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=11,
        color="black",
        clip_on=False
    )

    # =====================================================
    # Right axis: |k_dE| & |k_dN|
    # =====================================================
    ax2 = ax.twinx()

    ax2.plot(
        dates_initial,
        slope_dE,
        color=color_dE,
        linewidth=1.05,
        alpha=0.95,
        solid_capstyle="round",
        solid_joinstyle="round",
        zorder=2
    )

    ax2.plot(
        dates_initial,
        slope_dN,
        color=color_dN,
        linewidth=1.05,
        alpha=0.95,
        solid_capstyle="round",
        solid_joinstyle="round",
        zorder=2
    )

    # Do not use a unified ylabel; use separately colored right-axis labels instead
    ax2.set_ylabel("")

    ax2.text(
        1.085,
        0.62,
        r"$|k_{\mathrm{dE}}|$",
        transform=ax2.transAxes,
        rotation=90,
        ha="center",
        va="center",
        fontsize=12,
        color=color_dE,
        clip_on=False
    )

    ax2.text(
        1.085,
        0.50,
        "&",
        transform=ax2.transAxes,
        rotation=90,
        ha="center",
        va="center",
        fontsize=12,
        color="black",
        clip_on=False
    )

    ax2.text(
        1.085,
        0.37,
        r"$|k_{\mathrm{dN}}|$",
        transform=ax2.transAxes,
        rotation=90,
        ha="center",
        va="center",
        fontsize=12,
        color=color_dN,
        clip_on=False
    )

    # Display right-axis values as b, corresponding to b × 10^-3
    ax2.yaxis.set_major_formatter(scaled_tick_formatter(1e3))
    ax2.yaxis.set_major_locator(MaxNLocator(nbins=4))
    ax2.yaxis.set_minor_locator(AutoMinorLocator(5))

    # Place the right-axis scale factor at the bottom
    ax2.text(
        1.02,
        0.0,
        r"$\times 10^{-3}$",
        transform=ax2.transAxes,
        ha="left",
        va="top",
        fontsize=11,
        color="black",
        clip_on=False
    )

    # =====================================================
    # X axis: display 5 fixed major ticks
    # =====================================================
    major_ticks = pd.to_datetime([
        "2018-11-01",
        "2019-03-01",
        "2019-07-01",
        "2019-11-01",
        "2020-03-01"
    ])

    ax.set_xlim(plot_start, plot_end)
    ax.set_xticks(major_ticks)
    ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

    # =====================================================
    # Axis style
    # =====================================================
    for axis in [ax, ax2]:
        for spine in axis.spines.values():
            spine.set_visible(True)
            spine.set_linewidth(0.7)
            spine.set_color("black")

    ax.spines["right"].set_visible(False)
    ax2.spines["left"].set_visible(False)

    # Keep all left-axis and x-axis ticks black
    ax.tick_params(
        axis="both",
        which="major",
        direction="in",
        length=3.5,
        width=0.7,
        pad=2,
        top=False,
        right=False,
        bottom=True,
        left=True,
        colors="black"
    )

    ax.tick_params(
        axis="both",
        which="minor",
        direction="in",
        length=2.0,
        width=0.5,
        top=False,
        right=False,
        bottom=True,
        left=True,
        colors="black"
    )

    # Keep right-axis ticks and values black
    ax2.tick_params(
        axis="y",
        which="major",
        direction="in",
        length=3.5,
        width=0.7,
        pad=2,
        right=True,
        left=False,
        colors="black"
    )

    ax2.tick_params(
        axis="y",
        which="minor",
        direction="in",
        length=2.0,
        width=0.5,
        right=True,
        left=False,
        colors="black"
    )

    # Prevent duplicate x-axis display on the right axis
    ax2.tick_params(
        axis="x",
        which="both",
        bottom=False,
        top=False,
        labelbottom=False
    )

    ax.grid(False)
    ax2.grid(False)
    ax.margins(x=0.01)

    # Ensure the right-axis curves are not covered by the left-axis background
    ax.patch.set_visible(False)

    # =====================================================
    # Remove legend
    # =====================================================
    # Do not call ax.legend()

    # =====================================================
    # Four time boxes: draw according to the left-axis range
    # =====================================================
    y0, y1 = ax.get_ylim()
    yrange = y1 - y0

    box_y_bottom = y0 + 0.035 * yrange
    box_y_top = y1 - 0.02 * yrange

    blue_y_bottom = y0 + 0.035 * yrange
    blue_y_top = y0 + 0.65 * yrange

    add_four_boxes(
        ax=ax,
        y_bottom=box_y_bottom,
        y_top=box_y_top,
        blue_y_bottom=blue_y_bottom,
        blue_y_top=blue_y_top
    )

    # Prevent the time boxes from affecting the x range
    ax.set_xlim(plot_start, plot_end)

    # Leave space at the bottom for ×10^-2 and ×10^-3
    plt.subplots_adjust(
        left=0.10,
        right=0.90,
        bottom=0.30,
        top=1
    )

    # ------------------------
    # Save
    # ------------------------
    os.makedirs(output_dir, exist_ok=True)

    png_path = os.path.join(
        output_dir,
        "Slope_three_curves_double_y_replot.png"
    )

    pdf_path = os.path.join(
        output_dir,
        "Slope_three_curves_double_y_replot.pdf"
    )

    fig.savefig(png_path, dpi=600, bbox_inches="tight")
    fig.savefig(pdf_path, bbox_inches="tight")

    plt.close(fig)

    print(f"Saved: {png_path}")
    print(f"Saved: {pdf_path}")


# =========================================================
# 5. Main program
# =========================================================
if __name__ == "__main__":

    slope_csv = r"D:\a\master\Earthquake-US\Fig-Over-Output\Fig-4\Slope_three_curves_plot_data_after_A_or_V.csv"
    output_dir = r"D:\a\master\Earthquake-US\Fig-Over-Output\Fig-4"

    plot_slope_from_saved_csv(
        slope_csv=slope_csv,
        output_dir=output_dir,
        smooth=True,
        smooth_method="savgol",
        smooth_window=31,
        polyorder=2
    )
'''

In [ ]:
#The data of  (b)-(j)
'''
import os
import json
import glob
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler


# =========================
# 0. Basic functions
# =========================
def load_and_merge_data(csv_dir):
    """Read and merge all station CSV files in the folder"""
    files = sorted(glob.glob(os.path.join(csv_dir, "*.csv")))
    if not files:
        raise FileNotFoundError(f"No CSV files found in directory: {csv_dir}")

    dfs = []
    for file in files:
        df = pd.read_csv(file)
        df["site_id"] = os.path.splitext(os.path.basename(file))[0]
        dfs.append(df)

    df_merged = pd.concat(dfs, ignore_index=True)
    return df_merged


def preprocess_gnss_data(df, time_step=1):
    """
    Preprocess GNSS data:
    1) YYYYMMDD -> datetime
    2) Sample according to time_step
    3) Standardize dE / dN separately for each site
    4) Trim all sites to the same length
    5) Construct the A matrix and normalize it globally
    """
    df = df.copy()
    df["time"] = pd.to_datetime(df["YYYYMMDD"], format="%Y%m%d")

    sampled_times = np.sort(df["time"].unique())[::time_step]
    df_sampled = df[df["time"].isin(sampled_times)].copy()

    # Standardize dE / dN separately for each site
    for component in ["dE", "dN"]:
        std_col = f"{component}_standardized"
        df_sampled[std_col] = np.nan

        for site in sorted(df_sampled["site_id"].unique()):
            mask = df_sampled["site_id"] == site
            values = df_sampled.loc[mask, component].values.reshape(-1, 1)

            scaler = StandardScaler()
            df_sampled.loc[mask, std_col] = scaler.fit_transform(values).ravel()

    # Align the lengths of all sites
    site_lengths = df_sampled.groupby("site_id").size()
    min_length = site_lengths.min()

    valid_sites = sorted(site_lengths[site_lengths >= min_length].index.tolist())
    df_sampled = df_sampled[df_sampled["site_id"].isin(valid_sites)].copy()

    M = min_length
    sites = valid_sites
    N_T = 2 * len(sites)

    A = np.zeros((M, N_T), dtype=float)

    for i, site in enumerate(sites):
        site_data = (
            df_sampled[df_sampled["site_id"] == site]
            .sort_values("time")
            .iloc[:M]
        )
        A[:, 2 * i] = site_data["dE_standardized"].values
        A[:, 2 * i + 1] = site_data["dN_standardized"].values

    # Global normalization
    C0 = np.sum(A ** 2)
    if C0 <= 0:
        raise ValueError("A matrix energy C0 <= 0; normalization cannot be performed.")
    A_normalized = A / np.sqrt(C0)

    sampled_times = np.sort(df_sampled["time"].unique())[:M]
    return A_normalized, sampled_times, sites


def compute_time_modes(A):
    """
    Consistent with the logic of your original code: perform SVD on A.T,
    and finally take the temporal mode matrix time_modes = Vh.T
    """
    U, S, Vh = np.linalg.svd(A.T, full_matrices=False)
    eigenvalues = S ** 2
    time_modes = Vh.T
    return eigenvalues, time_modes, S


def rolling_std_fill(values, window=7):
    """Calculate the moving standard deviation and fill edge NaNs with the mean of non-NaN values"""
    values = np.asarray(values, dtype=float)
    out = pd.Series(values).rolling(window=window, center=True).std().values

    if np.all(np.isnan(out)):
        return np.zeros_like(values)

    valid_mean = np.nanmean(out)
    out = np.nan_to_num(out, nan=valid_mean)
    return out


def get_column_info(col_index, sites):
    """
    A matrix column index -> site / component
    0 -> site0 dE
    1 -> site0 dN
    2 -> site1 dE
    3 -> site1 dN
    """
    n_cols = 2 * len(sites)
    if col_index < 0 or col_index >= n_cols:
        raise ValueError(f"Column index {col_index} is out of range; valid range is 0 ~ {n_cols - 1}")

    site_index = col_index // 2
    component = "dE" if col_index % 2 == 0 else "dN"
    site_name = sites[site_index]
    return site_name, component


# =========================
# 1. Generate plotting data
# =========================
def prepare_plot_data(csv_dir, output_root, target_columns=None, window_size=30, step=1, time_step=1, rolling_window=7, start_window=0, end_window=None,):
    """
    Generate two types of plotting data:

    1) V_PC1:
       Save for each window:
       - date
       - pc1_score
       - moving_std

    2) A_columns:
       Save for each window and each specified column:
       - date
       - standardized_series
       - moving_std
       - col_index / site_name / component
    """
    if target_columns is None:
        target_columns = []

    output_root = Path(output_root)
    prepared_root = output_root / "prepared_plot_data"
    v_dir = prepared_root / "V_PC1"
    a_root = prepared_root / "A_columns"

    v_dir.mkdir(parents=True, exist_ok=True)
    a_root.mkdir(parents=True, exist_ok=True)

    # Read data and unify the time format
    df = load_and_merge_data(csv_dir)
    df["time"] = pd.to_datetime(df["YYYYMMDD"], format="%Y%m%d")
    all_times = np.sort(df["time"].unique())

    total_windows = len(range(0, len(all_times) - window_size + 1, step))
    if total_windows <= 0:
        raise ValueError("The window length exceeds the total time length; plotting data cannot be generated.")

    if end_window is None:
        end_window = total_windows - 1

    if start_window < 0 or end_window >= total_windows or start_window > end_window:
        raise ValueError(
            f"Invalid window range:
'''

In [ ]:
# (b)-(j) Time-series plots, plotted according to the trend, all oriented positive
'''
import os
import json
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import AutoMinorLocator, FuncFormatter

# =========================
# 0. Nature double-column 3×3 style
# =========================
MM_TO_INCH = 1 / 25.4
FIG_WIDTH = 183 * MM_TO_INCH
FIG_HEIGHT = 130 * MM_TO_INCH

EVENT_DATE = pd.to_datetime("2019-07-06")

TARGET_DATES = [
    "20190521", "20190528", "20190603",
    "20190608", "20190615", "20190703",
    "20190710", "20190730", "20190908",
]

GRID_ROWS = 3
GRID_COLS = 3

COLOR_V1 = "#1b5e9a"
COLOR_DE = "#b7652b"
COLOR_DN = "#4f8a5b"
COLOR_EVENT = "0.45"

# Whether to print the trend direction for each window
PRINT_TREND_INFO = True


def set_nature_style():
    plt.rcParams["font.family"] = "Arial"
    plt.rcParams["mathtext.fontset"] = "stix"

    plt.rcParams["font.size"] = 9
    plt.rcParams["axes.labelsize"] = 11
    plt.rcParams["axes.titlesize"] = 11
    plt.rcParams["xtick.labelsize"] = 9
    plt.rcParams["ytick.labelsize"] = 9
    plt.rcParams["legend.fontsize"] = 9

    plt.rcParams["axes.linewidth"] = 0.7
    plt.rcParams["xtick.major.width"] = 0.7
    plt.rcParams["ytick.major.width"] = 0.7
    plt.rcParams["xtick.minor.width"] = 0.5
    plt.rcParams["ytick.minor.width"] = 0.5

    plt.rcParams["xtick.major.size"] = 3.5
    plt.rcParams["ytick.major.size"] = 3.5
    plt.rcParams["xtick.minor.size"] = 2.0
    plt.rcParams["ytick.minor.size"] = 2.0

    plt.rcParams["pdf.fonttype"] = 42
    plt.rcParams["ps.fonttype"] = 42

    plt.rcParams["savefig.facecolor"] = "white"
    plt.rcParams["figure.facecolor"] = "white"
    plt.rcParams["axes.facecolor"] = "white"
    plt.rcParams["savefig.transparent"] = False


# =========================
# 1. Determine whether to reverse the sign based on the window trend
# =========================
def calc_linear_slope_by_date(dates, values):
    """
    Calculate the linear trend slope of a time series within a window.

    x uses the number of days relative to the start date of the window;
    y uses the original values.

    Return:
        slope: a in y = a x + b
    """
    dates = pd.to_datetime(dates)
    y = pd.to_numeric(values, errors="coerce").to_numpy(dtype=float)

    x = (dates - dates.min()).dt.total_seconds().to_numpy(dtype=float) / 86400.0

    mask = np.isfinite(x) & np.isfinite(y)

    if mask.sum() < 2:
        return np.nan

    x_valid = x[mask]
    y_valid = y[mask]

    # Prevent polyfit problems when all x values are identical
    if np.nanmax(x_valid) == np.nanmin(x_valid):
        x_valid = np.arange(len(y_valid), dtype=float)

    slope, intercept = np.polyfit(x_valid, y_valid, 1)
    return slope


def orient_series_by_window_trend(df, date_col, value_col):
    """
    Determine the plotting data according to the trend direction
    of value_col within each window.

    If the trend is upward, slope >= 0:
        plotting data = original data

    If the trend is downward, slope < 0:
        plotting data = -original data

    Return:
        df_out: DataFrame with an additional column used for plotting
        plot_col: new column name
        slope: trend slope of the original data
        sign: sign actually applied, +1 or -1
    """
    df_out = df.copy()

    slope = calc_linear_slope_by_date(
        dates=df_out[date_col],
        values=df_out[value_col],
    )

    if np.isfinite(slope) and slope < 0:
        sign = -1.0
    else:
        sign = 1.0

    plot_col = f"{value_col}_trend_oriented"
    df_out[plot_col] = sign * pd.to_numeric(df_out[value_col], errors="coerce")

    return df_out, plot_col, slope, sign


# =========================
# 2. Tick formatting
# =========================
def trim_formatter(x, pos):
    if abs(x) >= 1:
        s = f"{x:.1f}"
    elif abs(x) >= 0.1:
        s = f"{x:.2f}"
    elif abs(x) >= 0.01:
        s = f"{x:.3f}"
    else:
        s = f"{x:.4f}"
    return s.rstrip("0").rstrip(".")


def apply_fixed_ticks_3x3(ax1, ax2):
    """
    Unified y-axis ranges for the 3×3 plots:
    Left axis: V1
    Right axis: dE and dN
    """
    ax1.set_ylim(-0.9, 0.7)
    ax1.set_yticks([-0.50, 0.00, 0.50])

    ax2.set_ylim(-0.05, 0.06)
    ax2.set_yticks([-0.04, 0.00, 0.04])

    ax1.yaxis.set_major_formatter(FuncFormatter(trim_formatter))
    ax2.yaxis.set_major_formatter(FuncFormatter(trim_formatter))



# =========================
# 3. Axis formatting
# =========================
def format_axes_3x3(
    ax1,
    ax2,
    x_min,
    x_max,
    show_xlabel=False,
    show_left_label=False,
    show_right_label=False,
):
    ax1.set_xlim(x_min, x_max)
    ax1.grid(False)
    ax2.grid(False)

    # =========================
    # Display only 3 major ticks on the x axis:
    # midpoint -10 days, midpoint, midpoint +10 days
    # =========================
    x_min = pd.to_datetime(x_min)
    x_max = pd.to_datetime(x_max)
    
    # First calculate the midpoint, then normalize it to 00:00:00
    # of that day to avoid a 12:00 half-day offset
    x_mid = x_min + (x_max - x_min) / 2
    x_mid = x_mid.normalize()
    
    # Major ticks: midpoint -10 days, midpoint, midpoint +10 days
    major_ticks = pd.to_datetime([
        x_mid - pd.Timedelta(days=10),
        x_mid,
        x_mid + pd.Timedelta(days=10)
    ])
    
    ax1.set_xticks(major_ticks)
    
    # Generate minor ticks centered on x_mid as well,
    # avoiding misalignment with the major ticks
    left_days = int(np.floor((x_mid - x_min).total_seconds() / 86400))
    right_days = int(np.floor((x_max - x_mid).total_seconds() / 86400))
    
    minor_offsets = np.arange(-left_days, right_days + 1, 2)
    
    minor_ticks = pd.to_datetime([
        x_mid + pd.Timedelta(days=int(d))
        for d in minor_offsets
        if d not in [-10, 0, 10]   # Avoid overlap between minor and major ticks
    ])
    
    ax1.set_xticks(minor_ticks, minor=True)
    ax1.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))

    apply_fixed_ticks_3x3(ax1, ax2)

    ax1.yaxis.set_minor_locator(AutoMinorLocator(4))
    ax2.yaxis.set_minor_locator(AutoMinorLocator(4))

    # =========================
    # Left-axis ticks: keep black
    # =========================
    ax1.tick_params(
        axis="y",
        which="major",
        direction="in",
        left=show_left_label,
        right=False,
        labelleft=show_left_label,
        width=0.7,
        length=3.5,
        colors="black",
    )
    ax1.tick_params(
        axis="y",
        which="minor",
        direction="in",
        left=show_left_label,
        right=False,
        labelleft=False,
        width=0.5,
        length=2.0,
        colors="black",
    )

    # =========================
    # Right-axis ticks: keep black
    # =========================
    ax2.tick_params(
        axis="y",
        which="major",
        direction="in",
        left=False,
        right=show_right_label,
        labelright=show_right_label,
        width=0.7,
        length=3.5,
        colors="black",
    )
    ax2.tick_params(
        axis="y",
        which="minor",
        direction="in",
        left=False,
        right=show_right_label,
        labelright=False,
        width=0.5,
        length=2.0,
        colors="black",
    )

    # =========================
    # X-axis ticks: keep black
    # =========================
    ax1.tick_params(
        axis="x",
        which="major",
        direction="in",
        bottom=True,
        top=False,
        labelbottom=True,
        width=0.7,
        length=3.5,
        colors="black",
    )
    ax1.tick_params(
        axis="x",
        which="minor",
        direction="in",
        bottom=True,
        top=False,
        labelbottom=False,
        width=0.5,
        length=2.0,
        colors="black",
    )

    if show_xlabel:
        ax1.set_xlabel("Date (2019)", labelpad=2)
    else:
        ax1.set_xlabel("")

    # =========================
    # Left-axis label: V_1 in blue
    # while ticks and values remain black
    # =========================
    ax1.set_ylabel("")
    
    if show_left_label:
        ax1.text(
            -0.12,
            0.55,
            r"$V_1$",
            transform=ax1.transAxes,
            rotation=90,
            ha="center",
            va="center",
            fontsize=11,
            color=COLOR_V1,
            clip_on=False
        )

    # =========================
    # Right-axis labels: dE and dN colored separately
    # while right-axis ticks and values remain black
    # =========================
    ax2.set_ylabel("")

    if show_right_label:
        ax2.text(
            1.13,
            0.62,
            r"$\mathrm{dE}$",
            transform=ax2.transAxes,
            rotation=90,
            ha="center",
            va="center",
            fontsize=11,
            color=COLOR_DE,
            clip_on=False
        )

        ax2.text(
            1.13,
            0.50,
            "&",
            transform=ax2.transAxes,
            rotation=90,
            ha="center",
            va="center",
            fontsize=11,
            color="black",
            clip_on=False
        )

        ax2.text(
            1.13,
            0.36,
            r"$\mathrm{dN}$",
            transform=ax2.transAxes,
            rotation=90,
            ha="center",
            va="center",
            fontsize=11,
            color=COLOR_DN,
            clip_on=False
        )

    # =========================
    # Axis borders: keep black
    # =========================
    for side in ["left", "bottom", "top"]:
        ax1.spines[side].set_visible(True)
        ax1.spines[side].set_linewidth(0.7)
        ax1.spines[side].set_color("black")

    ax1.spines["right"].set_visible(False)

    for side in ["left", "bottom", "top"]:
        ax2.spines[side].set_visible(False)

    ax2.spines["right"].set_visible(True)
    ax2.spines["right"].set_linewidth(0.7)
    ax2.spines["right"].set_color("black")

    ax2.patch.set_visible(False)

    ax1.margins(x=0.01)


# =========================
# 4. Single-date subplot: V1 + dE + dN
# =========================
def plot_one_date_3x3(
    ax1,
    df_v,
    df_de,
    df_dn,
    date_key,
    show_xlabel=False,
    show_left_label=False,
    show_right_label=False,
    show_legend=False,
):
    ax2 = ax1.twinx()

    # =========================
    # Key modification:
    # Determine the trend directions of V1, dE, and dN
    # separately within each window.
    # If the trend is downward, multiply by -1 so that
    # the plotted trend is upward.
    # =========================
    df_v_plot, v_plot_col, slope_v, sign_v = orient_series_by_window_trend(
        df=df_v,
        date_col="date",
        value_col="pc1_score",
    )

    df_de_plot, de_plot_col, slope_de, sign_de = orient_series_by_window_trend(
        df=df_de,
        date_col="date",
        value_col="standardized_series",
    )

    df_dn_plot, dn_plot_col, slope_dn, sign_dn = orient_series_by_window_trend(
        df=df_dn,
        date_col="date",
        value_col="standardized_series",
    )

    if PRINT_TREND_INFO:
        print(
            f"{date_key} | "
            f"V1 slope={slope_v:.6g}, sign={sign_v:+.0f}; "
            f"dE slope={slope_de:.6g}, sign={sign_de:+.0f}; "
            f"dN slope={slope_dn:.6g}, sign={sign_dn:+.0f}"
        )

    dates_v = pd.to_datetime(df_v_plot["date"])
    dates_de = pd.to_datetime(df_de_plot["date"])
    dates_dn = pd.to_datetime(df_dn_plot["date"])

    x_min = min(dates_v.min(), dates_de.min(), dates_dn.min())
    x_max = max(dates_v.max(), dates_de.max(), dates_dn.max())

    line_v1, = ax1.plot(
        dates_v,
        df_v_plot[v_plot_col],
        color=COLOR_V1,
        linewidth=1.15,
        label=r"$V_1$",
        zorder=3,
    )

    line_de, = ax2.plot(
        dates_de,
        df_de_plot[de_plot_col],
        color=COLOR_DE,
        linewidth=1.05,
        label="dE",
        zorder=3,
    )

    line_dn, = ax2.plot(
        dates_dn,
        df_dn_plot[dn_plot_col],
        color=COLOR_DN,
        linewidth=1.05,
        label="dN",
        zorder=3,
    )

    format_axes_3x3(
        ax1=ax1,
        ax2=ax2,
        x_min=x_min,
        x_max=x_max,
        show_xlabel=show_xlabel,
        show_left_label=show_left_label,
        show_right_label=show_right_label,
    )

    ax1.set_title(pd.to_datetime(date_key).strftime("%Y-%m-%d"), pad=4)


    return ax2


# =========================
# 5. Read a single CSV
# =========================
def read_window_csv(file_path):
    df = pd.read_csv(file_path)
    df["date"] = pd.to_datetime(df["date"])
    return df


# =========================
# 6. Find dE / dN folders
# =========================
def find_component_dirs(a_input_root):
    subdirs = [p for p in Path(a_input_root).iterdir() if p.is_dir()]

    de_dirs = [p for p in subdirs if p.name.endswith("_dE")]
    dn_dirs = [p for p in subdirs if p.name.endswith("_dN")]

    if len(de_dirs) == 0:
        raise FileNotFoundError(f"No dE folder found under {a_input_root}")
    if len(dn_dirs) == 0:
        raise FileNotFoundError(f"No dN folder found under {a_input_root}")

    de_dir = sorted(de_dirs)[0]
    dn_dir = sorted(dn_dirs)[0]
    return de_dir, dn_dir


# =========================
# 7. Build an index by window_end date
# =========================
def build_date_map(file_list):
    """
    Build a mapping using window_end from the first row of each CSV:
    for example, '20190603' -> corresponding CSV file path
    """
    mapping = {}

    for f in file_list:
        df_head = pd.read_csv(f, nrows=1)

        if "window_end" not in df_head.columns:
            raise KeyError(f"{f} does not contain a window_end column and cannot be filtered by date")

        date_key = pd.to_datetime(df_head["window_end"].iloc[0]).strftime("%Y%m%d")
        mapping[date_key] = f

    return mapping


def get_files_by_target_dates(prepared_root, target_dates):
    prepared_root = Path(prepared_root)

    v_input_dir = prepared_root / "V_PC1"
    a_input_root = prepared_root / "A_columns"

    if not v_input_dir.exists():
        raise FileNotFoundError(f"Directory not found: {v_input_dir}")
    if not a_input_root.exists():
        raise FileNotFoundError(f"Directory not found: {a_input_root}")

    de_dir, dn_dir = find_component_dirs(a_input_root)

    v_files = sorted(glob.glob(str(v_input_dir / "*.csv")))
    de_files = sorted(glob.glob(str(de_dir / "*.csv")))
    dn_files = sorted(glob.glob(str(dn_dir / "*.csv")))

    if not v_files:
        raise FileNotFoundError(f"V_PC1 data not found: {v_input_dir}")
    if not de_files:
        raise FileNotFoundError(f"dE data not found: {de_dir}")
    if not dn_files:
        raise FileNotFoundError(f"dN data not found: {dn_dir}")

    v_map = build_date_map(v_files)
    de_map = build_date_map(de_files)
    dn_map = build_date_map(dn_files)

    selected = []

    for d in target_dates:
        key = pd.to_datetime(d).strftime("%Y%m%d")

        if key not in v_map:
            raise FileNotFoundError(f"No file with window_end={key} found in V_PC1")
        if key not in de_map:
            raise FileNotFoundError(f"No file with window_end={key} found in dE")
        if key not in dn_map:
            raise FileNotFoundError(f"No file with window_end={key} found in dN")

        selected.append({
            "date_key": key,
            "v_file": v_map[key],
            "de_file": de_map[key],
            "dn_file": dn_map[key],
        })

    return selected


# =========================
# 8. Plot 3×3 combined figure
# =========================
def plot_selected_dates_3x3(prepared_root, figure_root, target_dates):
    figure_root = Path(figure_root)
    figure_root.mkdir(parents=True, exist_ok=True)

    if len(target_dates) > GRID_ROWS * GRID_COLS:
        raise ValueError(
            f"A 3×3 figure supports at most {GRID_ROWS * GRID_COLS} dates; currently {len(target_dates)} were provided."
        )

    selected = get_files_by_target_dates(prepared_root, target_dates)

    fig, axes = plt.subplots(
        GRID_ROWS,
        GRID_COLS,
        figsize=(FIG_WIDTH, FIG_HEIGHT),
        sharex=False,
        sharey=False,
    )

    axes_flat = np.asarray(axes).ravel()

    for idx, item in enumerate(selected):
        row = idx // GRID_COLS
        col = idx % GRID_COLS
        ax = axes_flat[idx]

        df_v = read_window_csv(item["v_file"])
        df_de = read_window_csv(item["de_file"])
        df_dn = read_window_csv(item["dn_file"])

        show_left = (col == 0)
        show_right = (col == GRID_COLS - 1)
        show_xlabel = (row == GRID_ROWS - 1)
        show_legend = (idx == 0)

        plot_one_date_3x3(
            ax1=ax,
            df_v=df_v,
            df_de=df_de,
            df_dn=df_dn,
            date_key=item["date_key"],
            show_xlabel=show_xlabel,
            show_left_label=show_left,
            show_right_label=show_right,
            show_legend=show_legend,
        )

    for idx in range(len(selected), GRID_ROWS * GRID_COLS):
        axes_flat[idx].set_visible(False)

    plt.subplots_adjust(
        left=0.065,
        right=0.94,
        bottom=0.09,
        top=0.95,
        wspace=0.16,
        hspace=0.28,
    )

    date_tag = "_".join([pd.to_datetime(d).strftime("%Y%m%d") for d in target_dates])
    png_path = figure_root / f"combined_3x3_trend_oriented_{date_tag}.png"
    pdf_path = figure_root / f"combined_3x3_trend_oriented_{date_tag}.pdf"

    plt.savefig(png_path, dpi=600, bbox_inches="tight")
    plt.savefig(pdf_path, bbox_inches="tight")
    plt.close(fig)

    print(f"PNG saved: {png_path}")
    print(f"PDF saved: {pdf_path}")


# =========================
# 9. Main program
# =========================
if __name__ == "__main__":
    set_nature_style()

    output_root = Path(r"D:\a\master\Earthquake-US\Fig-Over-Output\Fig-4")
    prepared_root = output_root / "prepared_plot_data"
    figure_root = output_root / "figures_nature_3x3_selected_dates"

    figure_root.mkdir(parents=True, exist_ok=True)

    metadata_file = prepared_root / "metadata.json"
    if metadata_file.exists():
        with open(metadata_file, "r", encoding="utf-8") as f:
            metadata = json.load(f)
        print("metadata.json loaded")
        print(json.dumps(metadata, ensure_ascii=False, indent=2))

    plot_selected_dates_3x3(
        prepared_root=prepared_root,
        figure_root=figure_root,
        target_dates=TARGET_DATES,
    )

    print(f"\n3×3 combined figure output to: {figure_root}")
'''